In [42]:
# ============================================================
# 038_meeting_presentation_generator
# ============================================================
#
# Overview
# ----------------
# This notebook implements an LLM-driven workflow for generating
# meeting-ready, **editable PowerPoint (PPTX)** decks in Japanese.
# It collects user inputs via widgets, uses OpenAI to generate a
# presentation outline and structured slide specifications, uses Gemini
# to generate slide images, and assembles the final output as a **PPTX**
# (NOT PDF).
#
# The workflow emphasizes human-in-the-loop editing at key stages:
# users can review and modify the generated outline and slide JSON
# before image generation and PPTX export.
#
# Rendering policy (final deliverable = PPTX)
# ----------------
# - The PPTX is the canonical final artifact (editable text + replaceable images).
# - Text is always placed as PowerPoint text boxes (fully editable).
# - Images are generated by Gemini and inserted as follows:
#   1) Cover slide: a full-bleed 16:9 background image (Gemini).
#   2) Slides 2..N: a right-side image panel (Gemini) only.
#      The left side is a single text box containing the body text (~300 Japanese chars).
#
# Layout rules (strict)
# ----------------
# - Cover:
#   * Full-slide Gemini background image (16:9).
#   * Title and footer text are PowerPoint text boxes on top.
# - Slides 2..N:
#   * Two-column layout:
#       Left: ONE body text box (single box) with ~300 Japanese chars.
#       Right: ONE Gemini-generated image occupying the right panel.
#   * No geometric background patterns; consistent flat background color.
#   * No additional cards/blocks (avoid fragmentation).
#
# Typography rules (strict)
# ----------------
# - Title font size: 36 pt (all slides including cover)
# - Body font size: 18 pt (slides 2..N)
# - Japanese legibility is mandatory (no garbling; clear line spacing).
#
# Inputs / Outputs
# ----------------
# Inputs:
#   - Meeting counterpart name
#   - User name
#   - Summary of previous discussions
#   - Goal of current meeting
#   - Related research / notes
#   - OpenAI API key (via environment)
#   - Gemini API key (via environment)
#
# Outputs:
#   - One-page Japanese presentation outline (editable Markdown)
#   - Slide-by-slide JSON specification (editable, validated)
#     * JSON is the canonical source of slide text + image intent
#   - Generated Gemini images:
#     * cover image (full-bleed)
#     * right-panel images for slides 2..N
#   - Final assembled **PPTX** file in output/
#
# Structure
# ----------------
# Cell 01: Environment setup and imports
# Cell 02: Global configuration and design system definition (PPTX-first)
# Cell 03: Meeting context input form via ipywidgets
# Cell 04: OpenAI outline generation with editable display
# Cell 05: OpenAI slide JSON generation with validation (PPTX constraints)
# Cell 06: JSON editor and validation UI
# Cell 07: Gemini prompt enrichment (cover vs right-panel prompts)
# Cell 08: Generate Gemini images (cover + right panels)
# Cell 09: Image preview and review (cover/right-panel)
# Cell 10: PPTX assembly (text boxes + images)  ← PDFではなくPPTX
# Cell 11: Final output verification and export (PPTX + metadata)
#
# Notes
# ----------------
# - API keys must be provided via env.txt (never hardcoded)
# - All presentation content is generated in Japanese
# - Design system enforces 16:9 ratio, consistent background, no logos/flags
# - Human editing is expected between outline and image generation
# - JSON spec should include:
#   * slide title (Japanese)
#   * body text (Japanese, ~300 chars for slides 2..N; cover has no long body)
#   * visual_intent (diagram type & composition for Gemini; must NOT leak into body)
#   * image_prompt (English) tailored to:
#       - cover full-bleed OR right-panel image-only
# - Gemini model: configurable (default gemini-3-pro-image-preview)
# - This is a prototype; clarity and extensibility over production polish
# - No OCR dependency; structured JSON is the canonical text source


In [43]:
# ============================================================
# Cell 01 — Environment setup and imports (PPTX-first)
# ============================================================
# Overview:
#   Loads environment variables and imports libraries required across the notebook:
#   OpenAI, Gemini (supports both SDK styles), ipywidgets, PIL, JSON utilities,
#   and file I/O helpers.
#
# Notes:
#   - API keys are never hardcoded; always loaded from env.txt
#   - Keep imports minimal and aligned to PPTX final output (no PDF deps here)
# ============================================================

# --- Core Python libraries ---
import os
import json
import re
import io
import time
import base64
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Any, Tuple
from datetime import datetime

warnings.filterwarnings("ignore")

# --- Mandatory env loading ---
from dotenv import load_dotenv
PROJECT_DIR = Path(".").resolve()
ENV_PATH = PROJECT_DIR / "env.txt"
if not ENV_PATH.exists():
    raise FileNotFoundError(
        "env.txt not found. Please create env.txt in the working directory "
        "(or adjust ENV_PATH)."
    )

load_dotenv(str(ENV_PATH))

# --- Runtime LLM configuration (overridable later) ---
llm_provider = "OpenAI"
llm_model = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
llm_temperature = float(os.getenv("OPENAI_TEMPERATURE", "0.2"))  # 0.0でもOK。濃さが欲しければ0.2推奨
llm_max_tokens = int(os.getenv("OPENAI_MAX_TOKENS", "7000"))

# --- OpenAI client ---
try:
    from openai import OpenAI
    openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
except Exception as e:
    openai_client = None
    print(f"⚠️  WARNING: OpenAI client init failed: {type(e).__name__}: {e}")

# --- Gemini SDK (support both new and old styles) ---
# We don't fully init a client here; we just confirm imports are available.
_HAVE_GOOGLE_GENAI = False
_HAVE_GOOGLE_GENERATIVEAI = False

try:
    from google import genai as google_genai  # google-genai (new)
    _HAVE_GOOGLE_GENAI = True
except Exception:
    _HAVE_GOOGLE_GENAI = False

try:
    import google.generativeai as genai  # google-generativeai (older)
    _HAVE_GOOGLE_GENERATIVEAI = True
except Exception:
    _HAVE_GOOGLE_GENERATIVEAI = False

# --- Image handling ---
from PIL import Image, ImageDraw, ImageFont

# --- Jupyter widgets for user input ---
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML, clear_output

# --- Create output directory if not exists ---
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Verify API keys are loaded ---
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if not OPENAI_API_KEY:
    print("⚠️  WARNING: OPENAI_API_KEY not found in environment")
if not GEMINI_API_KEY and not GOOGLE_API_KEY:
    print("⚠️  WARNING: Neither GEMINI_API_KEY nor GOOGLE_API_KEY found in environment")

# --- Quick environment summary ---
print("✓ Environment setup complete (PPTX-first)")
print(f"✓ OpenAI provider: {llm_provider}")
print(f"✓ OpenAI model: {llm_model}")
print(f"✓ Temperature: {llm_temperature}")
print(f"✓ Max tokens: {llm_max_tokens}")
print(f"✓ Output directory: {OUTPUT_DIR}")
print(f"✓ Gemini SDK available: google-genai={_HAVE_GOOGLE_GENAI}, google-generativeai={_HAVE_GOOGLE_GENERATIVEAI}")


✓ Environment setup complete (PPTX-first)
✓ OpenAI provider: OpenAI
✓ OpenAI model: gpt-4o-mini
✓ Temperature: 0.2
✓ Max tokens: 7000
✓ Output directory: output
✓ Gemini SDK available: google-genai=True, google-generativeai=True


In [44]:
# ============================================================
# Cell 02 — Global configuration and design system definition (PPTX-final)
# ============================================================
# Overview:
#   Defines design constants used across the workflow to keep slides consistent.
#   Final artifact is PPTX:
#     - Cover: Gemini generates full-slide background image (optional).
#     - Non-cover: Gemini generates RIGHT-side image only (diagram/visual).
#     - All text (title + body) is rendered by PowerPoint (python-pptx).
#
# Outputs:
#   DESIGN_SYSTEM, IMAGE_GEN_CONFIG, SLIDE_SCHEMA (layout+blocks), PRESENTATION_SCHEMA
#   validate_* helpers
# ============================================================

import re
from typing import Any, Dict, List, Tuple

# ============================================================
# Design System (PPTX layout + typography)
# ============================================================

DESIGN_SYSTEM = {
    # --- Canvas ---
    "aspect_ratio": "16:9",
    "image_width": 1920,
    "image_height": 1080,

    # --- PPTX typography (your spec) ---
    "pptx_font_family": "Noto Sans JP",  # fallback handled in PPTX cell if not available
    "pptx_title_font_pt": 36,
    "pptx_body_font_pt": 18,

    # --- Slide layout grid (for PPTX placement guidance) ---
    # Use these as % for positioning (actual PPTX units computed later)
    "layout_grid": {
        "title_area": {"x": 0.06, "y": 0.06, "w": 0.88, "h": 0.12},   # top band
        "body_left":  {"x": 0.06, "y": 0.20, "w": 0.52, "h": 0.74},   # left text box
        "visual_right":{"x": 0.62, "y": 0.20, "w": 0.32, "h": 0.74},  # right image box
    },

    # --- Visual style tokens ---
    "colors": {
        "primary": "#0A211A",     # deep green-black
        "accent":  "#0AC985",     # teal
        "secondary": "#1F6F5C",   # mid green
        "bg": "#FFFFFF",          # solid white background for non-cover
        "muted": "#6B7280",       # neutral gray
        "card_bg": "#F6F8F7",     # very light gray-green (optional panels)
        "divider": "#E5E7EB",
    },

    # --- Consistency rules (non-cover) ---
    "non_cover_style_rules": [
        "Solid background only (pure white or very light gray-green).",
        "NO geometric patterns / NO polygons / NO hexagons / NO decorative background shapes.",
        "Title is always top-left, same margin and size across slides.",
        "Right side is visual-only (diagram/shape/icon), avoid paragraph text.",
    ],

    # --- Content guidelines ---
    "body_target_chars": (160, 320),  # you said ~300 chars; allow range
    "body_min_sentences": 3,
    "title_max_chars": 40,
}

# ============================================================
# Image Generation Configuration (Gemini)
# ============================================================

# We generate TWO kinds of images:
#  1) cover_background: full 1920x1080
#  2) right_visual: right-panel image only (still generate 16:9, but composition must keep right side clean)

IMAGE_GEN_CONFIG = {
    "model": "gemini-3-pro-image-preview",
    "temperature": 0.6,
    "max_output_tokens": 2048,

    # --- Global negative constraints to reduce style drift ---
    "global_prohibitions": [
        "No logos", "No flags", "No brand marks", "No watermarks",
        "No realistic human faces", "No photos", "No heavy gradients",
        "No decorative background patterns on non-cover slides",
    ],

    # --- Prompt prefix for NON-COVER right visuals ---
    # IMPORTANT: We ask Gemini to generate a clean visual that fits into the right panel.
    # Text rendering is risky; we restrict text to short axis labels only.
    "right_visual_prompt_prefix": (
        "Create a clean, modern consulting-style VISUAL for a presentation slide.\n"
        "Output image: 16:9 (1920x1080).\n"
        "This image will be placed ONLY on the RIGHT side of a slide, next to Japanese body text rendered in PowerPoint.\n"
        "Therefore:\n"
        "- DO NOT include a slide title.\n"
        "- DO NOT include paragraph text.\n"
        "- Only minimal labels are allowed (e.g., short axis labels, 1-3 words).\n"
        "- Use a solid, plain background (white or very light gray-green). NO patterns, NO polygons, NO decorative shapes.\n"
        "- Use flat vector shapes, thin lines, simple icons, and clear diagram structure.\n"
        "Color palette: deep green-black #0A211A, teal #0AC985, mid green #1F6F5C, neutral gray #6B7280.\n"
        "Style: card-like blocks with rounded corners and subtle shadow (very subtle), plenty of whitespace.\n"
    ),

    # --- Prompt prefix for COVER background ---
    "cover_background_prompt_prefix": (
        "Create a professional presentation COVER background image.\n"
        "Output image: 16:9 (1920x1080).\n"
        "The title and footer text will be added later in PowerPoint, so:\n"
        "- Do NOT include any text.\n"
        "- Keep a clean abstract tech/strategy feel.\n"
        "- You MAY use a very subtle geometric motif ONLY on cover (optional), but keep it minimal.\n"
        "Color palette: deep green-black #0A211A, teal #0AC985, mid green #1F6F5C, neutral gray #6B7280.\n"
        "Avoid logos, flags, brands, watermarks, faces, photos.\n"
    ),
}

# ============================================================
# Slide JSON Schema (layout+blocks, PPTX-friendly)
# ============================================================

_ALLOWED_LAYOUTS = {"cover", "one_column", "two_panel", "three_panel", "diagram"}
_ALLOWED_PANELS = {"full", "left", "middle", "right"}

SLIDE_SCHEMA = {
    "type": "object",
    "required": ["slide_number", "title", "content", "visual_intent", "image_prompt"],
    "properties": {
        "slide_number": {"type": "integer", "minimum": 1},
        "title": {"type": "string", "maxLength": DESIGN_SYSTEM["title_max_chars"]},
        "content": {
            "type": "object",
            "required": ["layout", "blocks"],
            "properties": {
                "layout": {"type": "string"},
                "blocks": {"type": "array"},
                "bullets": {"type": "array"},
                "notes": {"type": "string"},
            },
        },
        "visual_intent": {"type": "string"},
        "image_prompt": {"type": "string"},
    },
}

PRESENTATION_SCHEMA = {
    "type": "object",
    "required": ["metadata", "slides"],
    "properties": {
        "metadata": {"type": "object"},
        "slides": {"type": "array", "minItems": 1, "items": SLIDE_SCHEMA},
    },
}

# ============================================================
# Validation helpers (soft, blocks-aware)
# ============================================================

def _is_str(x: Any) -> bool:
    return isinstance(x, str)

def validate_slide_json_soft(slide: dict) -> Tuple[bool, List[str]]:
    errs: List[str] = []

    if not isinstance(slide, dict):
        return False, ["slide must be an object"]

    if not isinstance(slide.get("slide_number"), int) or slide["slide_number"] < 1:
        errs.append("slide_number must be a positive integer")

    title = slide.get("title")
    if not _is_str(title) or not title:
        errs.append("title must be a non-empty string")
    elif len(title) > DESIGN_SYSTEM["title_max_chars"]:
        errs.append(f"title exceeds {DESIGN_SYSTEM['title_max_chars']} chars")

    if not _is_str(slide.get("visual_intent")) or not slide.get("visual_intent"):
        errs.append("visual_intent must be a non-empty string")

    if not _is_str(slide.get("image_prompt")) or not slide.get("image_prompt"):
        errs.append("image_prompt must be a non-empty string")

    content = slide.get("content")
    if not isinstance(content, dict):
        errs.append("content must be an object")
        return (len(errs) == 0, errs)

    layout = content.get("layout")
    if layout not in _ALLOWED_LAYOUTS:
        errs.append(f"content.layout must be one of {_ALLOWED_LAYOUTS}")

    blocks = content.get("blocks", [])
    if not isinstance(blocks, list) or len(blocks) == 0:
        errs.append("content.blocks must be a non-empty array")

    # Cover-specific minimal rules
    if slide.get("slide_number") == 1:
        if layout != "cover":
            errs.append("cover slide must have layout='cover'")
        # cover blocks often: TITLE + FOOTER
        if isinstance(blocks, list) and len(blocks) < 2:
            errs.append("cover slide should have >=2 blocks (TITLE, FOOTER)")

    # Block checks
    if isinstance(blocks, list):
        for i, b in enumerate(blocks):
            if not isinstance(b, dict):
                errs.append(f"blocks[{i}] must be an object")
                continue
            panel = b.get("panel")
            if panel not in _ALLOWED_PANELS:
                errs.append(f"blocks[{i}].panel must be one of {_ALLOWED_PANELS}")
            if not _is_str(b.get("heading")):
                errs.append(f"blocks[{i}].heading must be a string")
            if not _is_str(b.get("body")):
                errs.append(f"blocks[{i}].body must be a string (can be empty for diagram-only panels)")

            # Raw newline in JSON strings is risky
            body = b.get("body", "")
            if isinstance(body, str) and "\n" in body:
                errs.append(f"blocks[{i}].body contains raw newline; use '\\\\n'")

    return (len(errs) == 0, errs)

def validate_presentation_json_soft(pres: dict) -> Tuple[bool, List[str]]:
    errs: List[str] = []
    if not isinstance(pres, dict):
        return False, ["presentation must be an object"]

    slides = pres.get("slides")
    if not isinstance(slides, list) or len(slides) == 0:
        return False, ["slides must be a non-empty array"]

    nums = []
    for si, s in enumerate(slides):
        ok, e = validate_slide_json_soft(s)
        if not ok:
            errs.append(f"Slide#{si+1}: " + "; ".join(e))
        n = s.get("slide_number")
        if isinstance(n, int):
            nums.append(n)

    if len(nums) != len(set(nums)):
        errs.append("slide_number values must be unique")

    return (len(errs) == 0, errs)

# ============================================================
# Consistency helper for prompts (non-cover)
# ============================================================

def non_cover_consistency_directive() -> str:
    rules = "\n".join([f"- {r}" for r in DESIGN_SYSTEM["non_cover_style_rules"]])
    return (
        "CONSISTENCY DIRECTIVE (NON-COVER):\n"
        f"{rules}\n"
        "If any instruction conflicts, prioritize this directive.\n"
    )

print("✓ Design system configuration loaded (PPTX-final)")
print(f"  - Canvas: {DESIGN_SYSTEM['image_width']}x{DESIGN_SYSTEM['image_height']} ({DESIGN_SYSTEM['aspect_ratio']})")
print(f"  - PPTX fonts: title={DESIGN_SYSTEM['pptx_title_font_pt']}pt, body={DESIGN_SYSTEM['pptx_body_font_pt']}pt")
print(f"  - Gemini model: {IMAGE_GEN_CONFIG['model']}")
print("  - Validators: validate_slide_json_soft(), validate_presentation_json_soft()")


✓ Design system configuration loaded (PPTX-final)
  - Canvas: 1920x1080 (16:9)
  - PPTX fonts: title=36pt, body=18pt
  - Gemini model: gemini-3-pro-image-preview
  - Validators: validate_slide_json_soft(), validate_presentation_json_soft()


In [45]:
# ============================================================
# Cell 03 — Meeting context input form via ipywidgets (hardened)
# ============================================================

import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import datetime

# ------------------------------------------------------------
# Widgets
# ------------------------------------------------------------
w_counterpart = widgets.Text(
    value='Keisuke Nakatsuka',
    description='相手名:',
    placeholder='例: 荒井 晃平',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='500px')
)

w_user_name = widgets.Text(
    value='',
    description='あなたの名前:',
    placeholder='例: 中塚 恵介',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='500px')
)

# Optional (recommended): deck title for PPTX
w_deck_title = widgets.Text(
    value='',
    description='資料タイトル:',
    placeholder='(任意) 例: CVCの価値をいかに経営に届けるか',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='700px')
)

w_previous_discussion = widgets.Textarea(
    value='',
    description='前回の議論:',
    placeholder='前回のミーティングで話し合った内容を簡潔にまとめてください',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='700px', height='120px')
)

w_meeting_goal = widgets.Textarea(
    value='',
    description='今回の目的:',
    placeholder='今回のミーティングで達成したい目標や議論したいテーマを記入してください',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='700px', height='120px')
)

w_research_notes = widgets.Textarea(
    value='',
    description='関連資料・メモ:',
    placeholder='(オプション) 関連する調査結果、参考資料、追加のメモなど',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='700px', height='120px')
)

w_output = widgets.Output()

w_submit = widgets.Button(
    description='入力内容を確定',
    button_style='success',  # primary は環境によって効かないことがある
    icon='check',
    layout=widgets.Layout(width='200px')
)

w_edit_unlock = widgets.Button(
    description='編集を再開（ロック解除）',
    button_style='warning',
    icon='unlock',
    layout=widgets.Layout(width='220px'),
    disabled=True
)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def _set_disabled(disabled: bool):
    for w in [w_counterpart, w_user_name, w_deck_title, w_previous_discussion, w_meeting_goal, w_research_notes]:
        w.disabled = disabled
    w_submit.disabled = disabled
    w_edit_unlock.disabled = not disabled

def _preview(text: str, n=80) -> str:
    t = (text or "").strip().replace("\n", " ")
    return t[:n] + ("..." if len(t) > n else "")

# ------------------------------------------------------------
# Handlers
# ------------------------------------------------------------
def on_submit_clicked(_):
    global meeting_context

    with w_output:
        clear_output()

        errors = []
        if not w_counterpart.value.strip():
            errors.append('相手名は必須項目です')
        if not w_user_name.value.strip():
            errors.append('あなたの名前は必須項目です')
        if not w_previous_discussion.value.strip():
            errors.append('前回の議論は必須項目です')
        if not w_meeting_goal.value.strip():
            errors.append('今回の目的は必須項目です')

        if errors:
            print('❌ 入力エラー:')
            for e in errors:
                print(f'  - {e}')
            return

        # Store validated data
        meeting_context = {
            "counterpart_name": w_counterpart.value.strip(),
            "user_name": w_user_name.value.strip(),
            "deck_title": w_deck_title.value.strip(),  # optional
            "previous_discussion": w_previous_discussion.value.strip(),
            "meeting_goal": w_meeting_goal.value.strip(),
            "research_notes": w_research_notes.value.strip(),
            "timestamp": datetime.now().isoformat(),
        }

        print('✓ ミーティング情報を確定しました\n')
        print('【確認】')
        print(f"相手名: {meeting_context['counterpart_name']}")
        print(f"あなたの名前: {meeting_context['user_name']}")
        if meeting_context.get("deck_title"):
            print(f"資料タイトル(任意): {meeting_context['deck_title']}")
        print(f"前回の議論: {_preview(meeting_context['previous_discussion'])}")
        print(f"今回の目的: {_preview(meeting_context['meeting_goal'])}")
        print('関連資料: ' + ('あり' if meeting_context['research_notes'] else 'なし'))
        print('\n次のセル (Cell 04) でアウトライン生成を実行できます')

        # Lock fields to prevent accidental drift
        _set_disabled(True)

def on_unlock_clicked(_):
    with w_output:
        clear_output()
        print("🔓 ロック解除しました。編集後、再度「入力内容を確定」を押してください。")
    _set_disabled(False)

w_submit.on_click(on_submit_clicked)
w_edit_unlock.on_click(on_unlock_clicked)

# ------------------------------------------------------------
# UI
# ------------------------------------------------------------
form_title = widgets.HTML(value='<h2>📝 ミーティング情報入力フォーム</h2>')
form_description = widgets.HTML(
    value=(
        '<p style="color: #555; margin-bottom: 20px;">'
        'プレゼンテーション生成に必要な情報を入力してください。<br>'
        '（本文はPowerPointで組むため、文章は具体的に書くほど品質が上がります）'
        '</p>'
    )
)

form_ui = widgets.VBox([
    form_title,
    form_description,
    widgets.HTML('<h3 style="margin-top: 20px;">基本情報</h3>'),
    w_counterpart,
    w_user_name,
    w_deck_title,
    widgets.HTML('<h3 style="margin-top: 30px;">ミーティング内容</h3>'),
    w_previous_discussion,
    w_meeting_goal,
    w_research_notes,
    widgets.HTML('<div style="margin-top: 20px;"></div>'),
    widgets.HBox([w_submit, w_edit_unlock]),
    w_output,
])

display(form_ui)

print('\n💡 ヒント:')
print('  - 「資料タイトル」は任意（入れるとPPTXの表紙タイトル生成で便利）')
print('  - 確定後は入力がロックされます（ズレ防止）')
print('  - 修正したい場合は「編集を再開」を押してください')



💡 ヒント:
  - 「資料タイトル」は任意（入れるとPPTXの表紙タイトル生成で便利）
  - 確定後は入力がロックされます（ズレ防止）
  - 修正したい場合は「編集を再開」を押してください


In [46]:
# ============================================================
# Cell 04 — OpenAI outline generation with editable display (PPTX-first)
# ============================================================
# Overview:
#   Generates a PPTX-first outline in Japanese:
#   - Cover: title/subtitle + full-slide Gemini background image direction
#   - Non-cover slides: left text box (≈300 Japanese chars) + right image direction (Gemini)
#   Provides an editable widget and confirmation workflow.
#
# Outputs:
#   presentation_outline (string, global)
#   outline_structured (dict, optional helper for Cell 05)
# ============================================================

import re
from openai import OpenAI
import ipywidgets as widgets
from IPython.display import display, clear_output

# ------------------------------------------------------------
# OpenAI client init
# ------------------------------------------------------------
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("✓ OpenAI client initialized")
print(f"  Model: {llm_model}")
print(f"  Temperature: {llm_temperature}")
print()

# ------------------------------------------------------------
# Prompt builder (PPTX-first)
# ------------------------------------------------------------
def build_outline_prompt(context: dict) -> str:
    deck_title = (context.get("deck_title") or "").strip()
    if not deck_title:
        deck_title = "プレゼンテーションタイトル案（必要に応じて編集）"

    prompt = f"""あなたは「経営向け・コンサル品質」のPowerPoint構成を作る専門家です。
以下のミーティング情報に基づき、PPTX最終出力を前提にした“スライド設計アウトライン”を日本語で作成してください。

【前提（重要）】
- 最終成果物は PowerPoint（PPTX）で編集可能なもの。
- 表紙:
  - 文字（タイトル/サブタイトル/日付/作成者）はPowerPoint側で配置する。
  - 背景は Geminiで「全画面の背景画像」を生成する想定。
- 表紙以外:
  - 左側：本文テキストをPowerPointで1ボックスに配置（約260〜340文字、である調、3〜4文）。
  - 右側：Geminiで生成する“右側パネル用画像”を配置（図解・アイコン・簡易チャート等）。
  - 右側画像には長文を入れない（軸ラベルや短い見出し程度は可）。
- 視覚要素（右側画像の指示）は、本文に混ぜず「右側ビジュアル指示」として分離して書くこと。

【ミーティング情報】
相手名: {context.get('counterpart_name','')}
あなたの名前: {context.get('user_name','')}

前回の議論:
{context.get('previous_discussion','')}

今回の目的:
{context.get('meeting_goal','')}
"""
    if context.get("research_notes"):
        prompt += f"\n関連資料・メモ:\n{context.get('research_notes','')}\n"

    prompt += f"""

【出力形式（Markdown固定）】
- 全体は 10〜12スライド相当（表紙含む）
- 各スライドは必ず次のテンプレに従うこと（見出し名も固定）:

# {deck_title}

## Slide 1: 表紙
- タイトル案: ...
- サブタイトル案: ...
- 右側ビジュアル指示: （ここは「表紙の背景」指示。抽象背景。幾何学模様なし、単色/淡いグラデのみ等）
- メモ: （任意）

## Slide 2: ...
- 目的: ...
- 主要メッセージ: ...
- 左本文（PPT本文・約300字）: ...
- 右側ビジュアル指示（Gemini画像）: ...
- 話すポイント（任意・1行）: ...

【品質要件】
- “総論の言い換え”は禁止。各スライドは必ず以下のうち2つ以上を含むこと：
  1) 因果/構造/分解（例：Why-How-What、Input/Output/Outcome、因果矢印）
  2) 設計案（例：KPI定義・運用・会議体・意思決定）
  3) 比較（Before/After、トレードオフ、2x2）
  4) リスク/論点（落とし穴、未解決の問い）
- 左本文は「である調」、3〜4文、約260〜340字を目安。
- 右側ビジュアル指示は図解の種類まで具体的に（例：2x2、マトリクス、フロー、タイムライン、KPIツリー等）。
- 右側ビジュアルは “背景幾何学模様なし” を基本（表紙以外）。
- 出力はMarkdownのみ。前置き説明は不要。
"""
    return prompt

# ------------------------------------------------------------
# Generate outline
# ------------------------------------------------------------
def generate_outline(context: dict) -> str:
    prompt = build_outline_prompt(context)

    print("🤖 OpenAIでアウトラインを生成中...\n")

    resp = openai_client.chat.completions.create(
        model=llm_model,
        messages=[
            {"role": "system", "content": "あなたはPPTXのスライド構成を設計する専門家です。出力はMarkdownのみ。"},
            {"role": "user", "content": prompt},
        ],
        temperature=llm_temperature,
        max_tokens=2600,
    )
    outline = (resp.choices[0].message.content or "").strip()

    print("✓ アウトライン生成完了")
    try:
        print(f"  トークン使用: {resp.usage.total_tokens}")
    except Exception:
        pass
    print()

    return outline

# ------------------------------------------------------------
# Optional: parse helper (best-effort) for later cells
# ------------------------------------------------------------
def parse_outline_to_structured(md: str) -> dict:
    """
    Best-effort parser that extracts per-slide blocks from the Markdown outline.
    Not strict; used only as a convenience.
    """
    structured = {"title": "", "slides": []}
    lines = md.splitlines()

    # title: first H1
    for ln in lines:
        if ln.strip().startswith("# "):
            structured["title"] = ln.strip()[2:].strip()
            break

    # slide sections
    slide_blocks = re.split(r"\n(?=##\s*Slide\s*\d+\s*:)", md)
    for blk in slide_blocks:
        m = re.search(r"##\s*Slide\s*(\d+)\s*:\s*(.+)", blk)
        if not m:
            continue
        sn = int(m.group(1))
        stitle = m.group(2).strip()
        item = {"slide_number": sn, "title": stitle}

        # capture bullet fields
        def _cap(key):
            mm = re.search(rf"-\s*{re.escape(key)}\s*:\s*(.+)", blk)
            return (mm.group(1).strip() if mm else "")

        item["purpose"] = _cap("目的")
        item["message"] = _cap("主要メッセージ")
        item["left_body"] = _cap("左本文（PPT本文・約300字）")
        if not item["left_body"]:
            item["left_body"] = _cap("左本文")  # fallback
        item["right_visual"] = _cap("右側ビジュアル指示（Gemini画像）")
        if not item["right_visual"]:
            item["right_visual"] = _cap("右側ビジュアル指示")
        item["cover_title"] = _cap("タイトル案")
        item["cover_subtitle"] = _cap("サブタイトル案")
        item["notes"] = _cap("話すポイント（任意・1行）")
        if not item["notes"]:
            item["notes"] = _cap("メモ")

        structured["slides"].append(item)

    # sort
    structured["slides"] = sorted(structured["slides"], key=lambda x: x.get("slide_number", 0))
    return structured

# ------------------------------------------------------------
# UI: editable display + confirm/regen
# ------------------------------------------------------------
if "meeting_context" not in globals():
    print("❌ エラー: meeting_contextが見つかりません")
    print("   Cell 03を実行して、ミーティング情報を入力してください")
else:
    generated_outline = generate_outline(meeting_context)

    w_outline_editor = widgets.Textarea(
        value=generated_outline,
        description="",
        layout=widgets.Layout(width="900px", height="520px"),
        style={"font_family": "monospace"},
    )

    w_outline_output = widgets.Output()

    w_outline_confirm = widgets.Button(
        description="アウトラインを確定",
        button_style="success",
        icon="check",
        layout=widgets.Layout(width="200px"),
    )

    w_outline_regenerate = widgets.Button(
        description="再生成",
        button_style="warning",
        icon="refresh",
        layout=widgets.Layout(width="150px"),
    )

    def on_confirm_outline(_):
        global presentation_outline, outline_structured

        with w_outline_output:
            clear_output()
            txt = w_outline_editor.value.strip()
            if not txt:
                print("❌ エラー: アウトラインが空です")
                return

            presentation_outline = txt
            outline_structured = parse_outline_to_structured(presentation_outline)

            print("✓ アウトラインを確定しました")
            print(f"  文字数: {len(presentation_outline)}")
            print(f"  パース済みslides数(参考): {len(outline_structured.get('slides', []))}")
            print("\n次のセル (Cell 05) でスライドJSON生成を実行できます")

    def on_regenerate_outline(_):
        with w_outline_output:
            clear_output()
            print("🔄 アウトラインを再生成中...")

        try:
            new_outline = generate_outline(meeting_context)
            w_outline_editor.value = new_outline
            with w_outline_output:
                clear_output()
                print("✓ 新しいアウトラインを生成しました。必要に応じて編集してください。")
        except Exception as e:
            with w_outline_output:
                clear_output()
                print(f"❌ 再生成エラー: {type(e).__name__}: {e}")

    w_outline_confirm.on_click(on_confirm_outline)
    w_outline_regenerate.on_click(on_regenerate_outline)

    outline_title = widgets.HTML(value="<h2>📋 プレゼンテーションアウトライン（PPTX最終）</h2>")
    outline_instructions = widgets.HTML(
        value=(
            '<p style="color:#555; margin-bottom: 14px;">'
            'このアウトラインは「左=本文(約300字)」「右=Gemini図解」のPowerPoint構成を前提にしています。<br>'
            '必要に応じて編集し、「アウトラインを確定」を押してください。'
            "</p>"
        )
    )

    display(widgets.VBox([
        outline_title,
        outline_instructions,
        w_outline_editor,
        widgets.HBox([w_outline_confirm, w_outline_regenerate]),
        w_outline_output,
    ]))

    print("\n💡 ヒント:")
    print("  - 表紙以外は『左本文』『右側ビジュアル指示』が揃っているか確認")
    print("  - 視覚要素が本文に混ざっていたら、右側ビジュアル指示へ移す")
    print("  - 確定後は presentation_outline / outline_structured に保存されます")


✓ OpenAI client initialized
  Model: gpt-4o-mini
  Temperature: 0.2

🤖 OpenAIでアウトラインを生成中...

✓ アウトライン生成完了
  トークン使用: 6178




💡 ヒント:
  - 表紙以外は『左本文』『右側ビジュアル指示』が揃っているか確認
  - 視覚要素が本文に混ざっていたら、右側ビジュアル指示へ移す
  - 確定後は presentation_outline / outline_structured に保存されます


In [47]:
# ============================================================
# Cell 05 — OpenAI slide JSON generation (PPTX-first spec)
#   - FIXED: enforce 260-340 chars for body_text
#   - FIXED: post-repair pass for any short body_text (<260)
# ============================================================

import json
from datetime import datetime

# ------------------------------------------------------------
# JSON Repair Helper (same as before)
# ------------------------------------------------------------
def _repair_json_newlines(s: str) -> str:
    out = []
    in_string = False
    escape = False
    for ch in s:
        if in_string:
            if escape:
                out.append(ch); escape = False; continue
            if ch == "\\":
                out.append(ch); escape = True; continue
            if ch == "\n":
                out.append("\\n"); continue
            if ch == '"':
                in_string = False; out.append(ch); continue
            out.append(ch)
        else:
            if ch == '"':
                in_string = True
                out.append(ch)
            else:
                out.append(ch)
    return "".join(out)

# ------------------------------------------------------------
# Prompt Builder (PPTX-first)
# ------------------------------------------------------------
def build_slide_json_prompt_pptx(outline: str, context: dict) -> str:
    deck_title = (context.get("deck_title") or "").strip() or "プレゼンテーションタイトル"

    prompt = f"""
あなたはプロフェッショナルなプレゼン設計者です。
以下のアウトライン（Markdown）を、PowerPoint最終出力のためのJSON仕様に変換してください。

【重要：最終成果物はPPTX】
- 表紙:
  - 背景のみ生成（全画面背景画像）。
  - タイトル/サブタイトル/日付/作成者はPPTで文字として配置する（画像に文字を描かない）。
- 表紙以外:
  - 左側：本文テキストはPPTで1つのテキストボックスに入れる（**必ず260〜340文字**、である調、**3〜4文**）。
  - 右側：画像は“図のみ”。長文は禁止。
  - 右側画像に入れて良い文字は「軸ラベル」「短い見出し」「短語（最大6語程度）」のみ。

【アウトライン（確定済み）】
{outline}

【ミーティングコンテキスト】
相手名: {context.get('counterpart_name','')}
あなたの名前: {context.get('user_name','')}
目的: {context.get('meeting_goal','')}

============================================================
【入力テキスト→JSON 変換ルール（最重要）】
============================================================
アウトライン内には次の箇条書きが出てくる：
- 目的: ...
- 主要メッセージ: ...
- 視覚要素: ...

ルール：
1) 「目的」「主要メッセージ」は ppt.body_text の材料にする。
2) 「視覚要素」は ppt.body_text に絶対に入れない（引用・要約も禁止）。
   - 視覚要素は image.right_panel_prompt もしくは cover_background_prompt のみに反映する。
3) 視覚要素に含まれる“図中ラベル”が必要な場合のみ、
   image.right_panel_labels に短語配列で入れてよい（例：["短期","中長期","データ/AI","エネルギー"]）。
   - 「マトリクス構造を示す」等の説明文はラベルに入れない。
4) **non-cover スライドの ppt.body_text は必ず260〜340字に収めること。**
   - 出力前に必ず自分で文字数を数え、260未満なら背景説明/示唆/次アクションを追加して増やす。
   - 340を超えるなら冗長部を削って調整する。
   - **3〜4文・である調を維持する。**
   - **視覚要素に触れないまま文字数を満たす。**

============================================================
【出力JSON形式（厳守）】
============================================================
必ず有効なJSONのみを出力すること。

{{
  "metadata": {{
    "title": "{deck_title}",
    "author": "{context.get('user_name','')}",
    "audience": "{context.get('counterpart_name','')}",
    "date": "{datetime.now().strftime('%Y-%m-%d')}",
    "total_slides": <number>
  }},
  "slides": [
    {{
      "slide_number": 1,
      "ppt": {{
        "title": "表紙タイトル（日本語）",
        "subtitle": "サブタイトル（日本語）",
        "body_text": ""
      }},
      "image": {{
        "type": "cover_background",
        "cover_background_prompt": "English prompt. Background only. No text. No logos. No geometric patterns. Subtle gradient allowed."
      }}
    }},
    {{
      "slide_number": 2,
      "ppt": {{
        "title": "スライドタイトル（日本語）",
        "subtitle": "",
        "body_text": "左本文（**必ず260〜340字**、である調、**3〜4文**）"
      }},
      "image": {{
        "type": "right_panel",
        "right_panel_prompt": "English prompt. Diagram-only for RIGHT PANEL. No paragraph text. Only short labels allowed.",
        "right_panel_labels": ["短いラベル1","短いラベル2"]
      }}
    }}
  ]
}}
"""
    return prompt.strip()

# ------------------------------------------------------------
# Soft Validation (PPTX-first) — NON-BLOCKING for length issues
#   - errors: fatal (block confirm)
#   - warnings: non-fatal (allow confirm)
# ------------------------------------------------------------
def validate_pptx_slide_json_soft(data: dict) -> tuple[bool, list[str], list[str]]:
    errors: list[str] = []
    warnings: list[str] = []

    if not isinstance(data, dict):
        return False, ["Top-level JSON must be an object."], []

    if not isinstance(data.get("metadata"), dict):
        errors.append("metadata must be an object.")

    slides = data.get("slides")
    if not isinstance(slides, list) or not slides:
        errors.append("slides must be a non-empty array.")
        return False, errors, warnings

    seen = set()
    for i, s in enumerate(slides):
        path = f"slides[{i}]"
        if not isinstance(s, dict):
            errors.append(f"{path} must be an object.")
            continue

        sn = s.get("slide_number")
        if not isinstance(sn, int) or sn < 1:
            errors.append(f"{path}.slide_number must be int >= 1.")
            continue

        if sn in seen:
            errors.append("slide_number must be unique.")
        seen.add(sn)

        ppt = s.get("ppt")
        img = s.get("image")

        # --- ppt checks (fatal) ---
        if not isinstance(ppt, dict):
            errors.append(f"{path}.ppt must be an object.")
        else:
            if not isinstance(ppt.get("title"), str) or not ppt.get("title"):
                errors.append(f"{path}.ppt.title must be a non-empty string.")

            if sn != 1:
                bt = ppt.get("body_text")
                if not isinstance(bt, str) or not bt.strip():
                    errors.append(f"{path}.ppt.body_text must be non-empty for non-cover slides.")
                else:
                    # NOTE: length is NON-FATAL now
                    bl = len(bt.strip())
                    if bl < 260:
                        warnings.append(f"{path}.ppt.body_text is short (<260 chars): {bl}")
                    elif bl > 340:
                        warnings.append(f"{path}.ppt.body_text is long (>340 chars): {bl}")

        # --- image checks (fatal) ---
        if not isinstance(img, dict):
            errors.append(f"{path}.image must be an object.")
        else:
            itype = img.get("type")
            if sn == 1:
                if itype != "cover_background":
                    errors.append(f"{path}.image.type must be 'cover_background' for slide 1.")
                if not isinstance(img.get("cover_background_prompt"), str) or not img.get("cover_background_prompt"):
                    errors.append(f"{path}.image.cover_background_prompt required.")
            else:
                if itype != "right_panel":
                    errors.append(f"{path}.image.type must be 'right_panel' for non-cover slides.")
                if not isinstance(img.get("right_panel_prompt"), str) or not img.get("right_panel_prompt"):
                    errors.append(f"{path}.image.right_panel_prompt required.")

                # labels optional (fatal only if wrong type)
                lbls = img.get("right_panel_labels", [])
                if lbls is not None and not isinstance(lbls, list):
                    errors.append(f"{path}.image.right_panel_labels must be an array if present.")
                if isinstance(lbls, list) and len(lbls) > 10:
                    warnings.append(f"{path}.image.right_panel_labels too many (>10): {len(lbls)}")

    ok = (len(errors) == 0)
    return ok, errors, warnings


# ------------------------------------------------------------
# Generate Slide JSON with OpenAI
# ------------------------------------------------------------
def generate_slide_json_pptx(outline: str, context: dict) -> dict:
    prompt = build_slide_json_prompt_pptx(outline, context)

    print("🤖 OpenAIでPPTX用スライドJSON仕様を生成中...\n")

    response = openai_client.chat.completions.create(
        model=llm_model,
        messages=[
            {"role": "system", "content": "You output ONLY valid JSON. No commentary."},
            {"role": "user", "content": prompt},
        ],
        temperature=globals().get("llm_temperature", 0.2),
        max_tokens=globals().get("llm_max_tokens", 6500),
        response_format={"type": "json_object"},
    )

    txt = (response.choices[0].message.content or "").strip()
    try:
        data = json.loads(txt)
    except json.JSONDecodeError:
        data = json.loads(_repair_json_newlines(txt))

    print("✓ JSON生成完了")
    try:
        print(f"  トークン使用: {response.usage.total_tokens}")
    except Exception:
        pass
    print(f"  スライド数: {len(data.get('slides', []))}\n")
    return data

# ------------------------------------------------------------
# Post-repair: expand only short body_text (<260)
# ------------------------------------------------------------
def expand_short_body_text_pptx(slide_data: dict) -> dict:
    slides = slide_data.get("slides", []) or []
    short_slide_numbers = []

    for s in slides:
        if s.get("slide_number") == 1:
            continue
        bt = (s.get("ppt", {}) or {}).get("body_text", "")
        if isinstance(bt, str) and len(bt.strip()) < 260:
            short_slide_numbers.append(s.get("slide_number"))

    if not short_slide_numbers:
        return slide_data

    print(f"🛠 本文が短いスライドを再調整します: {short_slide_numbers}")

    repair_prompt = (
        "You output ONLY valid JSON.\n"
        "Revise ONLY ppt.body_text for the specified slides.\n"
        "Do NOT change titles, subtitles, slide order, or any image prompts/labels.\n"
        "\n"
        "Constraints for each revised ppt.body_text:\n"
        "- Japanese 260-340 characters\n"
        "- 3-4 sentences\n"
        "- である調\n"
        "- Do NOT mention 視覚要素 (no quote, no summary)\n"
        "\n"
        f"TARGET_SLIDE_NUMBERS: {short_slide_numbers}\n"
        f"INPUT_JSON:\n{json.dumps(slide_data, ensure_ascii=False)}"
    )

    response = openai_client.chat.completions.create(
        model=llm_model,
        messages=[
            {"role": "system", "content": "You output ONLY valid JSON. No commentary."},
            {"role": "user", "content": repair_prompt},
        ],
        temperature=0.2,
        max_tokens=6500,
        response_format={"type": "json_object"},
    )

    txt = (response.choices[0].message.content or "").strip()
    try:
        repaired = json.loads(txt)
    except json.JSONDecodeError:
        repaired = json.loads(_repair_json_newlines(txt))

    print("✓ 本文の再調整完了\n")
    return repaired

# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------
def display_validation_results_pptx(slide_data: dict) -> None:
    print("🔍 JSON構造を検証中...\n")

    # ✅ 3戻りに対応
    ok, errors, warnings = validate_pptx_slide_json_soft(slide_data)

    md = slide_data.get("metadata", {}) or {}
    slides = slide_data.get("slides", []) or []

    if ok:
        print("✅ 検証OK: 致命エラーなし（PPTX生成へ進めます）\n")
    else:
        print("❌ 検証NG: 致命エラーあり（修正が必要）\n")

    print("【メタデータ】")
    print(f"  タイトル: {md.get('title','N/A')}")
    print(f"  作成者: {md.get('author','N/A')}")
    print(f"  対象: {md.get('audience','N/A')}")
    print(f"  スライド数: {len(slides)}\n")

    print("【スライド一覧】")
    for s in slides:
        sn = s.get("slide_number","?")
        title = (s.get("ppt",{}) or {}).get("title","")
        itype = (s.get("image",{}) or {}).get("type","")
        bt = (s.get("ppt",{}) or {}).get("body_text","")
        bt_len = len(bt.strip()) if isinstance(bt, str) else 0
        print(f"  #{sn}: {title}  | image={itype} | body_len={bt_len}")
    print()

    # ✅ warnings（非致命）を表示
    if warnings:
        print("⚠️ 警告（確定/生成は可能・品質改善推奨）:")
        for i, w in enumerate(warnings[:30], 1):
            print(f"  {i}. {w}")
        if len(warnings) > 30:
            print(f"  ...他 {len(warnings)-30} 件")
        print()

    # ✅ errors（致命）を表示
    if not ok:
        print("【致命エラー詳細】")
        for i, e in enumerate(errors, 1):
            print(f"  {i}. {e}")
        print()


# ------------------------------------------------------------
# Execute
# ------------------------------------------------------------
if "presentation_outline" not in globals():
    print("❌ エラー: presentation_outlineが見つかりません（Cell 04で確定してください）")
elif "meeting_context" not in globals():
    print("❌ エラー: meeting_contextが見つかりません（Cell 03を実行してください）")
else:
    try:
        slide_specifications = generate_slide_json_pptx(presentation_outline, meeting_context)

        # ---- NEW: auto-expand short body_text ----
        slide_specifications = expand_short_body_text_pptx(slide_specifications)

        display_validation_results_pptx(slide_specifications)

        json_path = OUTPUT_DIR / f"slide_specs_pptx_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(slide_specifications, f, ensure_ascii=False, indent=2)

        print(f"💾 JSONファイルを保存しました: {json_path}\n")

        # Preview first slide
        slides = slide_specifications.get("slides", []) or []
        if slides:
            s0 = slides[0]
            print("【サンプル: Slide 1】")
            print(f"  title: {(s0.get('ppt',{}) or {}).get('title','')}")
            print(f"  subtitle: {(s0.get('ppt',{}) or {}).get('subtitle','')}")
            cp = (s0.get("image",{}) or {}).get("cover_background_prompt","")
            print(f"  cover_background_prompt: {cp[:160]}...")
            print()

        print("✓ PPTX用スライドJSON仕様の生成が完了しました")
        print("次のステップ: Cell 06でJSON確認 → Cell 07/08で画像生成 → PPTX生成セルへ")

    except Exception as e:
        print(f"❌ 処理中にエラー: {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()

print("\n💡 ヒント:")
print("  - 表紙は cover_background_prompt（背景のみ・文字なし）")
print("  - 2枚目以降は right_panel_prompt（右パネル図のみ・長文禁止）")
print("  - 左本文は ppt.body_text（必ず260〜340字）にのみ入る")


🤖 OpenAIでPPTX用スライドJSON仕様を生成中...

✓ JSON生成完了
  トークン使用: 5758
  スライド数: 10

🛠 本文が短いスライドを再調整します: [2, 3, 4, 5, 6, 7, 8, 9, 10]
✓ 本文の再調整完了

🔍 JSON構造を検証中...

✅ 検証OK: 致命エラーなし（PPTX生成へ進めます）

【メタデータ】
  タイトル: Research OS 100 — Overview
  作成者: Keisuke Nakatsuka
  対象: Keisuke Nakatsukaの同僚
  スライド数: 10

【スライド一覧】
  #1: Research OS 100 — 概要  | image=cover_background | body_len=0
  #2: プロジェクトの目的  | image=right_panel | body_len=184
  #3: プロジェクトの範囲  | image=right_panel | body_len=153
  #4: 非目標の明確化  | image=right_panel | body_len=167
  #5: 進捗の概要  | image=right_panel | body_len=155
  #6: 技術環境  | image=right_panel | body_len=141
  #7: AIの使用に関する注意点  | image=right_panel | body_len=158
  #8: 長期的な目標  | image=right_panel | body_len=169
  #9: まとめと次のステップ  | image=right_panel | body_len=169
  #10: Q&A  | image=right_panel | body_len=170

⚠️ 警告（確定/生成は可能・品質改善推奨）:
  1. slides[1].ppt.body_text is short (<260 chars): 184
  2. slides[2].ppt.body_text is short (<260 chars): 153
  3. slides[3].ppt.body_text is short (<260 cha

In [48]:
# ============================================================
# Cell 06 — JSON editor and validation UI (PPTX-first spec)
# ============================================================
# Inputs:
#   slide_specifications (dict, from Cell 05 PPTX-first)
# Requires:
#   validate_pptx_slide_json_soft (from Cell 05 PPTX-first)
# Outputs:
#   confirmed_slide_specifications (dict)
# ============================================================

import json
import re
import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import datetime

# ------------------------------------------------------------
# Prerequisites
# ------------------------------------------------------------
if 'slide_specifications' not in globals():
    print('❌ エラー: slide_specifications が見つかりません')
    print('   Cell 05（PPTX-first差し替え版）を実行してください')
elif 'validate_pptx_slide_json_soft' not in globals():
    print('❌ エラー: validate_pptx_slide_json_soft が見つかりません')
    print('   Cell 05（PPTX-first差し替え版）を先に実行してください')
else:
    # ------------------------------------------------------------
    # Initialize editor with formatted JSON
    # ------------------------------------------------------------
    initial_json = json.dumps(slide_specifications, ensure_ascii=False, indent=2)

    w_json_editor = widgets.Textarea(
        value=initial_json,
        description='',
        layout=widgets.Layout(width='95%', height='620px'),
        style={'font_family': 'monospace', 'font_size': '12px'}
    )

    w_json_validation = widgets.Output(
        layout=widgets.Layout(
            border='1px solid #ddd',
            padding='10px',
            margin='10px 0',
            max_height='320px',
            overflow_y='auto'
        )
    )

    w_status_label = widgets.HTML(
        value='<p style="color: #888; font-style: italic;">編集中...（検証ボタンで構文・構造チェック）</p>'
    )

    w_validate_button = widgets.Button(
        description='検証',
        button_style='info',
        icon='check-circle',
        layout=widgets.Layout(width='120px')
    )

    w_confirm_button = widgets.Button(
        description='確定して次へ',
        button_style='success',
        icon='arrow-right',
        layout=widgets.Layout(width='150px'),
        disabled=True
    )

    w_reset_button = widgets.Button(
        description='リセット',
        button_style='warning',
        icon='undo',
        layout=widgets.Layout(width='120px')
    )

    w_save_backup_button = widgets.Button(
        description='バックアップ保存',
        button_style='',
        icon='save',
        layout=widgets.Layout(width='150px')
    )

    # ------------------------------------------------------------
    # Validation Logic (PPTX-first)
    # ------------------------------------------------------------
    def validate_current_json():
        json_text = w_json_editor.value.strip()
    
        # Step 1: Parse JSON syntax (fatal)
        try:
            parsed = json.loads(json_text)
        except json.JSONDecodeError as e:
            return (False, None, [f'JSONパースエラー (行 {e.lineno}): {e.msg}'], [])
    
        # Step 2: Structural validation (fatal + warning)
        ok, errors, warnings = validate_pptx_slide_json_soft(parsed)
        return (ok, parsed, errors, warnings)


    # ------------------------------------------------------------
    # Non-fatal QA hints (PPTX-first)
    # ------------------------------------------------------------
    _VISUAL_WORDS = [
        "視覚要素", "マトリクス", "フロー", "タイムライン", "比較表", "構造図",
        "図解", "チャート", "矢印", "2軸", "二軸"
    ]

    def _len_jp(s: str) -> int:
        return len((s or "").strip())

    def summarize_quality(parsed_data: dict) -> list[str]:
        hints: list[str] = []
        slides = parsed_data.get("slides", []) or []

        for s in slides:
            sn = s.get("slide_number", "?")
            ppt = s.get("ppt", {}) or {}
            img = s.get("image", {}) or {}
            title = ppt.get("title", "")

            if sn == 1:
                # Cover checks
                cp = (img.get("cover_background_prompt") or "")
                if img.get("type") != "cover_background":
                    hints.append(f"Slide#{sn}({title}): coverは image.type='cover_background' 推奨")
                if not cp:
                    hints.append(f"Slide#{sn}({title}): cover_background_prompt が空です")
                low = cp.lower()
                if "no text" not in low and "without text" not in low:
                    hints.append(f"Slide#{sn}({title}): cover_background_prompt に 'no text' を明記推奨（文字が画像に混入しやすい）")
                if "geometric" in low or "hexagon" in low or "polygon" in low:
                    hints.append(f"Slide#{sn}({title}): 背景の幾何学模様を避けたいなら geometric/hexagon/polygon は外す")
                continue

            # Non-cover checks
            bt = ppt.get("body_text", "") or ""
            bl = _len_jp(bt)

            if img.get("type") != "right_panel":
                hints.append(f"Slide#{sn}({title}): 非表紙は image.type='right_panel' 推奨")

            rp = (img.get("right_panel_prompt") or "")
            rpl = rp.lower()
            if "diagram-only" not in rpl:
                hints.append(f"Slide#{sn}({title}): right_panel_prompt に 'diagram-only' を入れると事故が減る")
            if "no paragraph text" not in rpl and "do not render paragraph text" not in rpl:
                hints.append(f"Slide#{sn}({title}): right_panel_prompt に 'no paragraph text' を明記推奨")

            # body length
            if bl < 220:
                hints.append(f"Slide#{sn}({title}): 左本文が短いかも（目安260〜340字、現在{bl}字）")
            if bl > 420:
                hints.append(f"Slide#{sn}({title}): 左本文が長いかも（目安260〜340字、現在{bl}字）")

            # Visual-element leakage heuristic
            if any(w in bt for w in _VISUAL_WORDS):
                hints.append(f"Slide#{sn}({title}): 左本文に“視覚要素っぽい語”が混入している可能性（本文から除去推奨）")

            # Labels sanity
            labels = img.get("right_panel_labels", [])
            if labels is not None and isinstance(labels, list) and len(labels) > 8:
                hints.append(f"Slide#{sn}({title}): right_panel_labels が多いかも（<=8推奨）")

        return hints

    # ------------------------------------------------------------
    # Event Handlers
    # ------------------------------------------------------------
    def on_validate_clicked(_):
        with w_json_validation:
            clear_output()
            print('🔍 検証中...\n')
    
            ok, parsed_data, errors, warnings = validate_current_json()
    
            if ok:
                print('✅ 構造検証OK（致命エラーなし）\n')
    
                md = parsed_data.get('metadata', {}) or {}
                slides = parsed_data.get('slides', []) or []
    
                print('【サマリー】')
                print(f"  タイトル: {md.get('title', 'N/A')}")
                print(f"  作成者: {md.get('author', 'N/A')}")
                print(f"  対象: {md.get('audience', 'N/A')}")
                print(f"  スライド数: {len(slides)}\n")
    
                print('【スライド一覧】')
                for s in slides:
                    sn = s.get("slide_number", "?")
                    ppt = s.get("ppt", {}) or {}
                    img = s.get("image", {}) or {}
                    t = ppt.get("title", "N/A")
                    it = img.get("type", "N/A")
                    bl = _len_jp(ppt.get("body_text","") or "")
                    print(f"  #{sn}: {t} | image={it} | body_len={bl}")
                print()
    
                # --- warnings from validator (non-fatal) ---
                if warnings:
                    print('⚠️  警告（確定は可能 / 品質改善推奨）:')
                    for w in warnings[:30]:
                        print(f"  - {w}")
                    if len(warnings) > 30:
                        print(f"  ...他 {len(warnings)-30} 件")
                    print()
    
                # --- existing QA hints (non-fatal) ---
                hints = summarize_quality(parsed_data)
                if hints:
                    print('🟡  事故防止QA（確定は可能 / 参考）:')
                    for h in hints[:20]:
                        print(f"  - {h}")
                    if len(hints) > 20:
                        print(f"  ...他 {len(hints)-20} 件")
                    print()
    
                print('✓ 「確定して次へ」で進めます（警告があってもOK）。')
                w_status_label.value = '<p style="color: #28a745; font-weight: bold;">✅ 致命エラーなし — 確定可能（警告は表示）</p>'
                w_confirm_button.disabled = False
    
            else:
                print('❌ 致命エラー（修正が必要）\n')
                for i, e in enumerate(errors, 1):
                    print(f"  {i}. {e}")
                print('\nJSONを修正してから再度検証してください。')
    
                w_status_label.value = '<p style="color: #dc3545; font-weight: bold;">❌ 致命エラー — 修正が必要</p>'
                w_confirm_button.disabled = True


    def on_confirm_clicked(_):
        global confirmed_slide_specifications
    
        ok, parsed_data, errors, warnings = validate_current_json()
        if not ok:
            with w_json_validation:
                clear_output()
                print('❌ エラー: 致命エラーがあるため確定できません（先に修正してください）\n')
                for i, e in enumerate(errors, 1):
                    print(f"  {i}. {e}")
            return
    
        confirmed_slide_specifications = parsed_data
    
        confirmed_path = OUTPUT_DIR / f"slide_specs_confirmed_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        with open(confirmed_path, 'w', encoding='utf-8') as f:
            json.dump(confirmed_slide_specifications, f, ensure_ascii=False, indent=2)
    
        with w_json_validation:
            clear_output()
            print('✅ スライドJSONを確定しました（警告があっても確定可能）\n')
            print(f'💾 保存先: {confirmed_path}\n')
    
            slides = confirmed_slide_specifications.get("slides", []) or []
            print(f'スライド数: {len(slides)}\n')
    
            if warnings:
                print('⚠️  確定時点の警告（参考）:')
                for w in warnings[:30]:
                    print(f"  - {w}")
                if len(warnings) > 30:
                    print(f"  ...他 {len(warnings)-30} 件")
                print()
    
            print('次のセル: 右パネル画像生成（Cell 07/08）→ PPTX生成セルへ')
    
        w_status_label.value = '<p style="color: #28a745; font-weight: bold;">✅ 確定完了 — 次のセルへ進めます</p>'
        w_confirm_button.disabled = True
        w_validate_button.disabled = True
        w_json_editor.disabled = True
        w_reset_button.disabled = True
        w_save_backup_button.disabled = True


    def on_reset_clicked(_):
        w_json_editor.value = initial_json
        w_status_label.value = '<p style="color: #888; font-style: italic;">リセット完了 — 元のJSONに戻しました</p>'
        w_confirm_button.disabled = True
        with w_json_validation:
            clear_output()
            print('🔄 元のJSONに戻しました')
            print('   検証ボタンをクリックして再度チェックしてください')

    def on_save_backup_clicked(_):
        json_text = w_json_editor.value.strip()
        try:
            parsed = json.loads(json_text)
            backup_path = OUTPUT_DIR / f"slide_specs_backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
            with open(backup_path, 'w', encoding='utf-8') as f:
                json.dump(parsed, f, ensure_ascii=False, indent=2)
            with w_json_validation:
                clear_output()
                print(f'💾 バックアップを保存しました: {backup_path}')
        except json.JSONDecodeError as e:
            with w_json_validation:
                clear_output()
                print(f'❌ JSONパースエラー: 保存できません (行 {e.lineno}): {e.msg}')

    w_validate_button.on_click(on_validate_clicked)
    w_confirm_button.on_click(on_confirm_clicked)
    w_reset_button.on_click(on_reset_clicked)
    w_save_backup_button.on_click(on_save_backup_clicked)

    # ------------------------------------------------------------
    # UI Layout
    # ------------------------------------------------------------
    editor_title = widgets.HTML(value='<h2>📝 スライドJSON編集・検証（PPTX-first）</h2>')

    editor_instructions = widgets.HTML(
        value=(
            '<div style="background:#f8f9fa;padding:15px;border-left:4px solid #007bff;margin-bottom:20px;">'
            '<p style="margin:0;color:#333;">'
            '<b>前提:</b> 最終成果物はPowerPoint。<br>'
            '表紙は「背景のみGemini」、文字はPPT。2枚目以降は「左本文=PPT」「右図=Gemini」。<br><br>'
            '<b>手順:</b><br>'
            '1) JSONを編集（ppt.body_text / image.* を中心に）<br>'
            '2) 「検証」で構文・構造チェック + 事故防止QA（警告）<br>'
            '3) 問題なければ「確定して次へ」<br><br>'
            '<b>注意:</b> 視覚要素の説明文は本文に入れない（右パネルprompt側へ）'
            '</p>'
            '</div>'
        )
    )

    button_row = widgets.HBox(
        [w_validate_button, w_confirm_button, w_reset_button, w_save_backup_button],
        layout=widgets.Layout(margin='15px 0')
    )

    editor_ui = widgets.VBox([
        editor_title,
        editor_instructions,
        w_json_editor,
        button_row,
        w_status_label,
        w_json_validation
    ])

    display(editor_ui)

    print()
    print('💡 編集後に「検証」→問題なければ「確定して次へ」で進めます。')



💡 編集後に「検証」→問題なければ「確定して次へ」で進めます。


In [49]:
# ============================================================
# Cell 07 — Gemini setup + prompt builder (PPTX-first, WHITE BACKGROUND)
#   - Cover: full-slide background image (white-based, no text)
#   - Non-cover: diagram asset ONLY (15cm x 13cm), SOLID WHITE background
#   - Images must contain NO PPT text blocks
#   - Minimum labels allowed for diagrams (Japanese only)
#   - NO transparency / NO checkerboard
# ============================================================

import os
import json
import re
from datetime import datetime

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
GEMINI_IMAGE_MODEL = globals().get("GEMINI_IMAGE_MODEL", "gemini-3-pro-image-preview")

# Palette (diagram elements only)
BCAP_ACCENT = "#0AC985"   # teal
BCAP_DARK   = "#0A211A"   # deep dark
NEUTRAL_GRAY = "#6B7280"

# ------------------------------------------------------------
# Diagram asset size (15cm x 13cm → aspect 15:13)
# ------------------------------------------------------------
DIAGRAM_ASSET_W_PX = 1500
DIAGRAM_ASSET_H_PX = 1300

# ------------------------------------------------------------
# DESIGN SYSTEM (WHITE BACKGROUND — FINAL)
# ------------------------------------------------------------
DESIGN_SYSTEM_TEXT = f"""
GLOBAL IMAGE RULES (MUST FOLLOW):
- OUTPUT FORMAT: PNG
- BACKGROUND: PURE SOLID WHITE (#FFFFFF)
  * No transparency.
  * No checkerboard.
  * No texture, no gradient.
- Canvas size (DIAGRAM ASSET): {DIAGRAM_ASSET_W_PX}x{DIAGRAM_ASSET_H_PX} pixels (aspect 15:13).
- Style: modern consulting diagram asset, minimal, clean, crisp vector-like strokes.
- Palette (diagram elements only):
  * strokes/text: dark {BCAP_DARK}
  * highlights: accent teal {BCAP_ACCENT}
  * secondary strokes: neutral {NEUTRAL_GRAY}
- Shapes: rounded rectangles, thin lines, clean arrows, simple outline icons.
- Text policy:
  * This is a diagram asset for PPTX (title/body handled in PPT).
  * Japanese labels allowed ONLY when necessary (minimum).
  * Do NOT mix languages.
  * Long sentences are forbidden. Use short nouns/keywords only.
  * No title text, no footer text.
- ABSOLUTE:
  * NO LOGOS / NO WATERMARKS / NO BRAND MARKS
  * NO FLAGS / NO REALISTIC FACES
  * NO FRAMES or borders around the whole image
- Output: ONE single PNG image only.
""".strip()

# ------------------------------------------------------------
# Gemini client init
# ------------------------------------------------------------
def _ensure_api_key_present():
    if not (os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")):
        raise RuntimeError("Neither GEMINI_API_KEY nor GOOGLE_API_KEY is set.")

def _init_gemini_client():
    try:
        from google import genai
        client = genai.Client(api_key=os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY"))
        return ("google_genai", client)
    except Exception:
        pass

    try:
        import google.generativeai as genai
        genai.configure(api_key=os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY"))
        return ("google_generativeai", genai)
    except Exception:
        pass

    raise RuntimeError("No supported Gemini SDK found.")

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def _safe_int(x, default=0):
    try:
        return int(x)
    except Exception:
        return default

def _extract_left_body_text(slide: dict, max_chars: int = 420) -> str:
    content = (slide.get("content") or {}) or {}
    blocks = (content.get("blocks") or []) or []
    left_blocks = [b for b in blocks if (b.get("panel") or "") == "left"]
    use_blocks = left_blocks if left_blocks else blocks

    chunks = []
    for b in use_blocks:
        body = (b.get("body") or "").strip()
        if body:
            chunks.append(body)

    text = "\n".join(chunks).replace("\\n", "\n").strip()
    if len(text) > max_chars:
        text = text[:max_chars].rstrip() + "…"
    return text

def _extract_visual_intent(slide: dict) -> str:
    return (slide.get("visual_intent") or "").strip()

def _extract_right_labels_jp(slide: dict) -> list[str]:
    content = (slide.get("content") or {}) or {}
    blocks = (content.get("blocks") or []) or []
    labels = []

    for b in blocks:
        h = (b.get("heading") or "")
        body = (b.get("body") or "")
        if "図中ラベル" in h and body:
            for t in body.replace("、", ",").split(","):
                t = t.strip()
                if t:
                    labels.append(t)

    # fallback: from visual_intent
    if not labels:
        vi = _extract_visual_intent(slide)
        for token in re.split(r"[、,\n/・\s]+", vi):
            token = token.strip()
            if any(ord(c) > 127 for c in token) and len(token) <= 8:
                labels.append(token)
            if len(labels) >= 6:
                break

    return list(dict.fromkeys(labels))[:8]

# ------------------------------------------------------------
# COVER prompt (white background, text-free)
# ------------------------------------------------------------
def _cover_background_prompt(slide: dict) -> str:
    base = (slide.get("image_prompt") or "").strip()
    if not base:
        base = "Clean abstract consulting cover background."

    return f"""
GLOBAL COVER IMAGE RULES:
- Canvas: 1920x1080
- Background: solid white (#FFFFFF)
- Style: modern consulting, minimal, elegant
- ABSOLUTE: No text, no logos, no watermarks
- Leave negative space top-left for PPT title

COVER INTENT:
{base}
""".strip()

# ------------------------------------------------------------
# DIAGRAM prompt (WHITE BACKGROUND)
# ------------------------------------------------------------
def _right_panel_diagram_prompt(slide: dict) -> str:
    body = _extract_left_body_text(slide)
    vi = _extract_visual_intent(slide)
    labels = _extract_right_labels_jp(slide)

    labels_block = ""
    if labels:
        labels_block = "MINIMUM JAPANESE LABELS:\n" + "\n".join([f"- {l}" for l in labels])

    return f"""
{DESIGN_SYSTEM_TEXT}

DIAGRAM GOAL:
- Create a diagram that DIRECTLY reflects the body text and intent below.
- Structure must match causal flow / comparison / loop described.
- Do NOT invent unrelated visuals.

REFERENCE BODY (for structure only):
{body}

VISUAL INTENT (guidance only):
{vi}

{labels_block}

COMPOSITION:
- 2–5 main nodes max
- Clear arrows / relationships
- Minimal text, Japanese only
- No title text
""".strip()

# ------------------------------------------------------------
# Build prompts for Cell 08
# ------------------------------------------------------------
if 'confirmed_slide_specifications' not in globals():
    print('❌ confirmed_slide_specifications not found')
else:
    _ensure_api_key_present()
    mode, gemini_client = _init_gemini_client()
    gemini_model = gemini_client

    slides = confirmed_slide_specifications.get("slides", [])
    enriched_prompts = []

    for i, s in enumerate(slides):
        sn = _safe_int(s.get("slide_number"), i + 1)
        title = (s.get("title") or "").strip()

        if sn == 1:
            prompt = _cover_background_prompt(s)
            kind = "cover_background"
            w, h = 1920, 1080
        else:
            prompt = _right_panel_diagram_prompt(s)
            kind = "diagram_asset"
            w, h = DIAGRAM_ASSET_W_PX, DIAGRAM_ASSET_H_PX

        enriched_prompts.append({
            "slide_number": sn,
            "title": title,
            "kind": kind,
            "prompt": prompt,
            "target_width_px": w,
            "target_height_px": h,
            "background": "white"
        })

        print(f"✓ slide #{sn} prompt built ({kind}) size={w}x{h}")

    path = OUTPUT_DIR / f"enriched_prompts_whitebg_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(enriched_prompts, f, ensure_ascii=False, indent=2)

    print(f"\n💾 saved: {path}")
    print("→ 次は Cell 08 で画像生成")


✓ slide #1 prompt built (cover_background) size=1920x1080
✓ slide #2 prompt built (diagram_asset) size=1500x1300
✓ slide #3 prompt built (diagram_asset) size=1500x1300
✓ slide #4 prompt built (diagram_asset) size=1500x1300
✓ slide #5 prompt built (diagram_asset) size=1500x1300
✓ slide #6 prompt built (diagram_asset) size=1500x1300
✓ slide #7 prompt built (diagram_asset) size=1500x1300
✓ slide #8 prompt built (diagram_asset) size=1500x1300
✓ slide #9 prompt built (diagram_asset) size=1500x1300
✓ slide #10 prompt built (diagram_asset) size=1500x1300

💾 saved: output/enriched_prompts_whitebg_20260205_133918.json
→ 次は Cell 08 で画像生成


In [50]:
# ============================================================
# Cell 08 — Generate slide images for all slides (Gemini, PPTX-first, WHITE BACKGROUND)
# ============================================================
# Overview:
#   Generates images using Gemini for PPTX-first workflow:
#   - Cover (slide 1): full-slide background image (NO TEXT), white-based.
#   - Non-cover (slide 2+): diagram illustration ASSET (WHITE background, NO transparency).
#       - Target size: 15cm x 13cm  -> aspect 15:13 (default 1500 x 1300 from Cell 07)
#   Saves PNG files to OUTPUT_DIR with timestamped filenames.
#
# Inputs / Outputs:
#   Inputs:  enriched_prompts (list[dict], from Cell 07)  <-- (NOTE: name aligned to Cell 07)
#   Outputs: generated_slide_images (list[dict]) (global)
#
# Notes:
#   - If BOTH GEMINI_API_KEY and GOOGLE_API_KEY are set: ALWAYS uses GEMINI_API_KEY.
#   - Supports both google-genai and google-generativeai SDKs.
#   - Post-process:
#       - For diagram_asset: force WHITE background and resize to target size.
#       - We do NOT attempt background removal. Everything is white by design.
# ============================================================

import os
import time
import json
import base64
from datetime import datetime
from pathlib import Path

from PIL import Image

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
GEMINI_IMAGE_MODEL = globals().get("GEMINI_IMAGE_MODEL", "gemini-3-pro-image-preview")

# Prefer sizes coming from Cell 07 prompts; fallback to 1500x1300 (15:13)
DEFAULT_ASSET_W = 1500
DEFAULT_ASSET_H = 1300

OUTPUT_DIR = globals().get("OUTPUT_DIR", Path("output"))
if not isinstance(OUTPUT_DIR, Path):
    OUTPUT_DIR = Path(str(OUTPUT_DIR))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# ------------------------------------------------------------
# Key preference helpers (forced GEMINI_API_KEY if present)
# ------------------------------------------------------------
def _pick_api_key_prefer_gemini() -> tuple[str, str]:
    if GEMINI_API_KEY:
        return ("GEMINI_API_KEY", GEMINI_API_KEY)
    if GOOGLE_API_KEY:
        return ("GOOGLE_API_KEY", GOOGLE_API_KEY)
    raise RuntimeError("Neither GEMINI_API_KEY nor GOOGLE_API_KEY is set. Check env.txt and load_dotenv.")

def _force_env_key_selection(chosen_name: str, chosen_value: str) -> None:
    if chosen_name == "GEMINI_API_KEY":
        os.environ["GEMINI_API_KEY"] = chosen_value
        os.environ["GOOGLE_API_KEY"] = ""
    elif chosen_name == "GOOGLE_API_KEY":
        os.environ["GOOGLE_API_KEY"] = chosen_value
        os.environ["GEMINI_API_KEY"] = ""
    else:
        raise ValueError(f"Unknown key name: {chosen_name}")

# ------------------------------------------------------------
# Gemini client init (supports multiple SDK styles)
# ------------------------------------------------------------
def _init_gemini_client():
    key_name, api_key = _pick_api_key_prefer_gemini()
    _force_env_key_selection(key_name, api_key)

    # Candidate A: google-genai (new)
    try:
        from google import genai  # type: ignore
        client = genai.Client(api_key=api_key)
        return ("google_genai", client, key_name)
    except Exception:
        pass

    # Candidate B: google-generativeai (older)
    try:
        import google.generativeai as genai  # type: ignore
        genai.configure(api_key=api_key)
        return ("google_generativeai", genai, key_name)
    except Exception:
        pass

    raise RuntimeError("No supported Gemini SDK found. Install `google-genai` or `google-generativeai`.")

def _gemini_generate_image(mode: str, client_or_genai, prompt: str):
    if mode == "google_genai":
        client = client_or_genai
        return client.models.generate_content(
            model=GEMINI_IMAGE_MODEL,
            contents=[prompt],
        )

    if mode == "google_generativeai":
        genai = client_or_genai
        model = genai.GenerativeModel(GEMINI_IMAGE_MODEL)
        return model.generate_content([prompt])

    raise RuntimeError(f"Unknown Gemini mode: {mode}")

def _extract_png_bytes_from_response(mode: str, resp) -> bytes:
    # google-genai style
    if mode == "google_genai":
        try:
            cand0 = resp.candidates[0]
            parts = getattr(cand0.content, "parts", []) or []
            for p in parts:
                inline = getattr(p, "inline_data", None)
                if inline and getattr(inline, "data", None):
                    data = inline.data
                    if isinstance(data, str):
                        return base64.b64decode(data)
                    if isinstance(data, (bytes, bytearray)):
                        return bytes(data)
        except Exception:
            pass

    # google-generativeai style
    if mode == "google_generativeai":
        try:
            cand0 = resp.candidates[0]
            parts = getattr(cand0.content, "parts", []) or []
            for p in parts:
                inline = getattr(p, "inline_data", None)
                if inline and getattr(inline, "data", None):
                    data = inline.data
                    if isinstance(data, str):
                        return base64.b64decode(data)
                    if isinstance(data, (bytes, bytearray)):
                        return bytes(data)
        except Exception:
            pass

        try:
            parts = getattr(resp, "parts", []) or []
            for p in parts:
                inline = getattr(p, "inline_data", None)
                if inline and getattr(inline, "data", None):
                    data = inline.data
                    if isinstance(data, str):
                        return base64.b64decode(data)
                    if isinstance(data, (bytes, bytearray)):
                        return bytes(data)
        except Exception:
            pass

    raise ValueError("Could not extract image bytes from Gemini response.")

def _retry(fn, max_tries=3, base_sleep=2.0):
    last_err = None
    for attempt in range(1, max_tries + 1):
        try:
            return fn()
        except Exception as e:
            last_err = e
            sleep_s = base_sleep * (2 ** (attempt - 1))
            print(f"    ⚠️ retry {attempt}/{max_tries} after error: {type(e).__name__}: {e}")
            time.sleep(sleep_s)
    raise last_err

# ------------------------------------------------------------
# Post-processing helpers (WHITE BACKGROUND ENFORCEMENT)
# ------------------------------------------------------------
def _force_white_background_and_resize(png_path: Path, out_path: Path, target_w: int, target_h: int) -> None:
    """
    Ensures:
      - final image is RGB (no alpha)
      - background is pure white (#FFFFFF)
      - resized to target_w x target_h
    If input has alpha, it is composited onto white.
    """
    with Image.open(png_path) as im:
        if im.mode in ("RGBA", "LA"):
            bg = Image.new("RGBA", im.size, (255, 255, 255, 255))
            im = Image.alpha_composite(bg, im.convert("RGBA")).convert("RGB")
        else:
            im = im.convert("RGB")

        im = im.resize((target_w, target_h), resample=Image.LANCZOS)
        im.save(out_path, format="PNG", optimize=True)

# ------------------------------------------------------------
# Main
# ------------------------------------------------------------
if 'enriched_prompts' not in globals():
    print('❌ エラー: enriched_prompts が見つかりません')
    print('   Cell 07 (白背景版) を実行してプロンプトを構築してください')
else:
    run_ts = datetime.now().strftime("%Y%m%d_%H%M%S")

    mode, gemini_client, key_name_used = _init_gemini_client()
    print(f"✅ Gemini client initialized ({mode}), model={GEMINI_IMAGE_MODEL}")
    print(f"🔑 API key used: {key_name_used} (forced preference: GEMINI_API_KEY if present)")
    print(f"🕒 Run timestamp: {run_ts}")
    print()

    generated_slide_images = []
    total = len(enriched_prompts)

    print(f"🎨 {total}件の画像を生成中...")
    print()

    for i, item in enumerate(enriched_prompts):
        slide_num = int(item.get("slide_number", i + 1))
        title = item.get("title", "")
        kind = item.get("kind", "diagram_asset")
        prompt = item.get("prompt", "")

        target_w = int(item.get("target_width_px") or DEFAULT_ASSET_W)
        target_h = int(item.get("target_height_px") or DEFAULT_ASSET_H)

        raw_filename = f"gemini_raw_{run_ts}_{slide_num:03d}_{kind}.png"
        out_filename = f"slide_{run_ts}_{slide_num:03d}_{kind}.png"
        raw_path = OUTPUT_DIR / raw_filename
        out_path = OUTPUT_DIR / out_filename

        print(f"[{i+1}/{total}] Slide #{slide_num}: {title} ({kind})  target={target_w}x{target_h}")

        try:
            def _call():
                return _gemini_generate_image(mode, gemini_client, prompt)

            resp = _retry(_call, max_tries=3, base_sleep=2.0)
            png_bytes = _extract_png_bytes_from_response(mode, resp)

            with open(raw_path, "wb") as f:
                f.write(png_bytes)

            # Enforce white background + size
            if kind == "cover_background":
                # keep cover at native size if prompt generates 1920x1080, but still enforce white + RGB
                # If you want strict 1920x1080 always, set target_w/target_h accordingly in Cell 07.
                with Image.open(raw_path) as im:
                    if im.mode in ("RGBA", "LA"):
                        bg = Image.new("RGBA", im.size, (255, 255, 255, 255))
                        im = Image.alpha_composite(bg, im.convert("RGBA")).convert("RGB")
                    else:
                        im = im.convert("RGB")
                    im.save(out_path, format="PNG", optimize=True)
            else:
                # diagram_asset: strict size + white background
                _force_white_background_and_resize(raw_path, out_path, target_w, target_h)

            generated_slide_images.append({
                "run_timestamp": run_ts,
                "api_key_used": key_name_used,
                "slide_number": slide_num,
                "title": title,
                "kind": kind,
                "prompt_length": len(prompt or ""),
                "raw_image_path": str(raw_path),
                "image_path": str(out_path),
                "filename": out_filename,
                "status": "success",
                "error": None,
                "background": "white"
            })

            print(f"  ✓ saved: {out_filename}")

        except Exception as e:
            print(f"  ❌ error: {type(e).__name__}: {e}")
            generated_slide_images.append({
                "run_timestamp": run_ts,
                "api_key_used": key_name_used,
                "slide_number": slide_num,
                "title": title,
                "kind": kind,
                "prompt_length": len(prompt or ""),
                "raw_image_path": str(raw_path) if raw_path else None,
                "image_path": None,
                "filename": None,
                "status": "error",
                "error": str(e),
                "background": "unknown"
            })

        print()

    success_count = sum(1 for x in generated_slide_images if x["status"] == "success")
    error_count = sum(1 for x in generated_slide_images if x["status"] == "error")

    print("=" * 60)
    print("【画像生成サマリー】")
    print(f"  成功: {success_count}/{total}")
    print(f"  エラー: {error_count}")
    print()

    meta_path = OUTPUT_DIR / f"generated_images_meta_{run_ts}_whitebg.json"
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(generated_slide_images, f, ensure_ascii=False, indent=2)

    print(f"💾 メタデータ保存: {meta_path}")
    print()
    if success_count > 0:
        print("✅ 次のステップ:")
        print("  - Cell 10 (PPTX) で、cover_background を表紙背景に、diagram_asset を右側(15cm x 13cm)に配置してください。")


✅ Gemini client initialized (google_genai), model=gemini-3-pro-image-preview
🔑 API key used: GEMINI_API_KEY (forced preference: GEMINI_API_KEY if present)
🕒 Run timestamp: 20260205_133921

🎨 10件の画像を生成中...

[1/10] Slide #1:  (cover_background)  target=1920x1080
  ✓ saved: slide_20260205_133921_001_cover_background.png

[2/10] Slide #2:  (diagram_asset)  target=1500x1300
  ✓ saved: slide_20260205_133921_002_diagram_asset.png

[3/10] Slide #3:  (diagram_asset)  target=1500x1300
  ✓ saved: slide_20260205_133921_003_diagram_asset.png

[4/10] Slide #4:  (diagram_asset)  target=1500x1300
  ✓ saved: slide_20260205_133921_004_diagram_asset.png

[5/10] Slide #5:  (diagram_asset)  target=1500x1300
  ✓ saved: slide_20260205_133921_005_diagram_asset.png

[6/10] Slide #6:  (diagram_asset)  target=1500x1300
  ✓ saved: slide_20260205_133921_006_diagram_asset.png

[7/10] Slide #7:  (diagram_asset)  target=1500x1300
  ✓ saved: slide_20260205_133921_007_diagram_asset.png

[8/10] Slide #8:  (diagram_asset

In [51]:
# ============================================================
# Cell 09 — Image preview and review (PPTX-first: cover/right kinds)
# ============================================================
# Overview:
#   Displays generated images in an interactive widget preview UI.
#   This PPTX-first version supports:
#     - cover_background images (full-slide, NO TEXT)
#     - right_panel images (diagram, left area blank, NO TEXT)
#
# Inputs / Outputs:
#   Inputs:  generated_slide_images (list, from Cell 08)
#            PNG files in OUTPUT_DIR
#   Outputs: Interactive preview UI with navigation
#            confirmed_images_ready (bool, global)
# ============================================================

from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output, Image as IPyImage

# ------------------------------------------------------------
# Prerequisites
# ------------------------------------------------------------
if 'generated_slide_images' not in globals():
    print('❌ エラー: generated_slide_images が見つかりません')
    print('   Cell 08 を実行して画像を生成してください')
else:
    # Filter successful images with existing files
    available_images = []
    for img in (generated_slide_images or []):
        if img.get('status') != 'success':
            continue
        p = img.get('image_path')
        if not p:
            continue
        pp = Path(p)
        if pp.exists() and pp.is_file():
            available_images.append(img)

    if not available_images:
        print('❌ 表示可能な画像がありません（status=success かつファイル存在が必要です）')
        print('   Cell 08で画像生成が成功しているか、OUTPUT_DIR配下にPNGがあるか確認してください')
    else:
        # Sort by slide_number, then kind (cover first), then filename
        def _kind_rank(k: str) -> int:
            k = (k or "").strip()
            if k == "cover_background":
                return 0
            if k == "right_panel":
                return 1
            return 9

        def _sort_key(d):
            sn = int(d.get('slide_number', 0) or 0)
            kind = d.get('image_kind', '') or ''
            fn = str(d.get('filename', '') or '')
            return (sn, _kind_rank(kind), fn)

        available_images = sorted(available_images, key=_sort_key)

        # Track state
        current_index = [0]  # mutable closure
        confirmed_images_ready = False  # global flag

        # ------------------------------------------------------------
        # Widgets
        # ------------------------------------------------------------
        w_title = widgets.HTML('<h2>🖼️ 画像プレビュー（PPTX-first）</h2>')

        w_info_label = widgets.HTML()
        w_meta_label = widgets.HTML()

        w_image_display = widgets.Output(
            layout=widgets.Layout(
                width='980px',
                height='560px',
                border='2px solid #ddd',
                padding='6px'
            )
        )

        w_prev_btn = widgets.Button(
            description='◀ 前へ',
            button_style='info',
            layout=widgets.Layout(width='110px')
        )
        w_next_btn = widgets.Button(
            description='次へ ▶',
            button_style='info',
            layout=widgets.Layout(width='110px')
        )
        w_confirm_all_btn = widgets.Button(
            description='✓ 全て確認完了',
            button_style='success',
            layout=widgets.Layout(width='160px')
        )

        w_jump = widgets.BoundedIntText(
            value=1,
            min=1,
            max=len(available_images),
            step=1,
            description='移動:',
            layout=widgets.Layout(width='170px')
        )
        w_jump_btn = widgets.Button(
            description='Go',
            button_style='',
            layout=widgets.Layout(width='60px')
        )

        w_status_out = widgets.Output(
            layout=widgets.Layout(
                border='1px solid #eee',
                padding='10px',
                margin='10px 0',
                max_height='160px',
                overflow_y='auto'
            )
        )

        # ------------------------------------------------------------
        # Render function
        # ------------------------------------------------------------
        def show_slide(index: int):
            index = max(0, min(index, len(available_images) - 1))
            current_index[0] = index

            img_data = available_images[index]
            slide_num = img_data.get('slide_number', '?')
            title = img_data.get('title', '')
            kind = img_data.get('image_kind', '') or ''
            filename = Path(img_data.get('image_path')).name
            run_ts = img_data.get('run_timestamp', '')

            kind_label = kind
            if kind == "cover_background":
                kind_label = "cover_background（表紙 背景）"
            elif kind == "right_panel":
                kind_label = "right_panel（右側 図）"

            # Info label
            w_info_label.value = (
                f'<p style="text-align:center;font-size:14px;color:#666;margin:0;">'
                f'{index + 1} / {len(available_images)} 枚目'
                f'</p>'
            )

            # Meta label
            w_meta_label.value = (
                f'<div style="text-align:center;color:#333;">'
                f'<div style="font-size:16px;font-weight:600;">'
                f'スライド #{slide_num}: {title}'
                f'</div>'
                f'<div style="font-size:13px;color:#444;">kind: {kind_label}</div>'
                f'<div style="font-size:12px;color:#666;">file: {filename}'
                + (f' / run: {run_ts}' if run_ts else '')
                + '</div>'
                f'</div>'
            )

            # Image display
            with w_image_display:
                clear_output(wait=True)
                p = Path(img_data['image_path'])
                display(IPyImage(filename=str(p)))

            # Buttons state
            w_prev_btn.disabled = (index == 0)
            w_next_btn.disabled = (index == len(available_images) - 1)

            # Jump widget sync
            w_jump.value = index + 1

        # ------------------------------------------------------------
        # Handlers
        # ------------------------------------------------------------
        def on_prev(_):
            show_slide(current_index[0] - 1)

        def on_next(_):
            show_slide(current_index[0] + 1)

        def on_jump(_):
            show_slide(int(w_jump.value) - 1)

        def on_confirm_all(_):
            global confirmed_images_ready
            confirmed_images_ready = True
            with w_status_out:
                clear_output()
                print('✅ 全画像の確認が完了しました')
                print(f'   確認済み画像数: {len(available_images)} 枚')
                print()
                print('次のステップ: （PPTX最終）PPTX組み立てセルへ進めます')

        w_prev_btn.on_click(on_prev)
        w_next_btn.on_click(on_next)
        w_jump_btn.on_click(on_jump)
        w_confirm_all_btn.on_click(on_confirm_all)

        # Layout
        nav_row = widgets.HBox(
            [w_prev_btn, w_next_btn, w_jump, w_jump_btn, w_confirm_all_btn],
            layout=widgets.Layout(justify_content='center', margin='10px 0', gap='8px')
        )

        preview_ui = widgets.VBox([
            w_title,
            w_info_label,
            w_meta_label,
            w_image_display,
            nav_row,
            w_status_out
        ])

        display(preview_ui)

        # Show first image
        show_slide(0)

        # Summary text
        cover_cnt = sum(1 for x in available_images if (x.get("image_kind") == "cover_background"))
        right_cnt = sum(1 for x in available_images if (x.get("image_kind") == "right_panel"))
        other_cnt = len(available_images) - cover_cnt - right_cnt

        print()
        print('💡 ヒント:')
        print('  - 「前へ/次へ」または番号指定で移動できます')
        print('  - 表紙は cover_background（文字はPPTで載せる想定）')
        print('  - 2枚目以降は right_panel（左側は本文18ptをPPTで載せる想定）')
        print(f'  - 画像内訳: cover={cover_cnt}, right={right_cnt}, other={other_cnt}')
        print('  - 気になるものがあれば Cell 06でJSON修正 → Cell 07/08で再生成')



💡 ヒント:
  - 「前へ/次へ」または番号指定で移動できます
  - 表紙は cover_background（文字はPPTで載せる想定）
  - 2枚目以降は right_panel（左側は本文18ptをPPTで載せる想定）
  - 画像内訳: cover=0, right=0, other=10
  - 気になるものがあれば Cell 06でJSON修正 → Cell 07/08で再生成


In [57]:
# ============================================================
# Cell 10 — PPTX assembly (FINAL OUTPUT, PPTX-first) — REPLACEMENT (ppt.* schema)
# ============================================================

from pathlib import Path
from datetime import datetime

# --- Ensure python-pptx is available ---
try:
    from pptx import Presentation
    from pptx.util import Inches, Pt, Cm
    from pptx.dml.color import RGBColor
    from pptx.enum.shapes import MSO_SHAPE
    from pptx.oxml.ns import qn
except ModuleNotFoundError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-pptx"])
    from pptx import Presentation
    from pptx.util import Inches, Pt, Cm
    from pptx.dml.color import RGBColor
    from pptx.enum.shapes import MSO_SHAPE
    from pptx.oxml.ns import qn

# ------------------------------------------------------------
# Prerequisites
# ------------------------------------------------------------
if 'confirmed_slide_specifications' not in globals():
    print('❌ エラー: confirmed_slide_specifications が見つかりません')
    print('   Cell 06でJSONを確定してください')
elif 'generated_slide_images' not in globals():
    print('❌ エラー: generated_slide_images が見つかりません')
    print('   Cell 08で画像生成を行ってください')
else:
    OUTPUT_DIR = globals().get("OUTPUT_DIR", Path("output"))
    if not isinstance(OUTPUT_DIR, Path):
        OUTPUT_DIR = Path(str(OUTPUT_DIR))
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # ------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------
    def _hex_to_rgb(hex_str: str) -> RGBColor:
        hs = (hex_str or "").strip().lstrip("#")
        if len(hs) != 6:
            return RGBColor(0, 0, 0)
        return RGBColor(int(hs[0:2], 16), int(hs[2:4], 16), int(hs[4:6], 16))

    def _safe_int(x, default=0):
        try:
            return int(x)
        except Exception:
            return default

    def _set_run_font_meiryo(run, size_pt: int, bold: bool = False, color: RGBColor | None = None):
        run.font.name = "Meiryo UI"
        run.font.size = Pt(size_pt)
        run.font.bold = bool(bold)
        if color is not None:
            run.font.color.rgb = color
        # East Asian font mapping (important for Japanese)
        try:
            rPr = run._element.get_or_add_rPr()
            rFonts = rPr.get_or_add_rFonts()
            rFonts.set(qn("w:eastAsia"), "Meiryo UI")
        except Exception:
            pass

    def _set_paragraph_font_meiryo(paragraph, size_pt: int, bold: bool = False, color: RGBColor | None = None):
        paragraph.font.name = "Meiryo UI"
        paragraph.font.size = Pt(size_pt)
        paragraph.font.bold = bool(bold)
        if color is not None:
            paragraph.font.color.rgb = color
        try:
            pPr = paragraph._p.get_or_add_pPr()
            defRPr = pPr.get_or_add_defRPr()
            rFonts = defRPr.get_or_add_rFonts()
            rFonts.set(qn("w:eastAsia"), "Meiryo UI")
        except Exception:
            pass

    def _build_image_map(images: list[dict]) -> dict[int, list[dict]]:
        """
        slide_number -> list of candidate image records (success & exists)
        We keep all kinds, and choose later by priority.
        """
        m: dict[int, list[dict]] = {}
        for it in (images or []):
            if it.get("status") != "success":
                continue
            sn = _safe_int(it.get("slide_number"), 0)
            p = it.get("image_path")
            if sn <= 0 or not p:
                continue
            pp = Path(p)
            if not (pp.exists() and pp.is_file()):
                continue
            m.setdefault(sn, []).append({**it, "image_path": str(pp)})
        return m

    def _pick_image_for_slide(sn: int, slide_spec: dict, image_map: dict[int, list[dict]]) -> str | None:
        """
        Pick best image path for this slide using both:
          - generated_slide_images.kind (if present)
          - slide_spec.image.type (cover_background/right_panel etc.)
        """
        cands = image_map.get(sn, []) or []
        if not cands:
            return None

        spec_img_type = ((slide_spec.get("image") or {}).get("type") or "").strip()

        # Priority lists
        if sn == 1:
            priorities = [
                "cover_bg", "cover_background", "cover-background",
                "cover", "background", "cover_background_image",
                spec_img_type,  # e.g., "cover_background"
            ]
        else:
            priorities = [
                "diagram_asset", "right_panel", "right-panel",
                "diagram", "right",
                spec_img_type,  # e.g., "right_panel"
            ]

        # Normalize
        def norm(x: str) -> str:
            return (x or "").strip().lower().replace(" ", "_").replace("-", "_")

        pri = [norm(x) for x in priorities if x]
        # sort candidates by whether their kind hits priority order
        best = None
        best_rank = 10**9

        for it in cands:
            k = norm(it.get("kind") or it.get("type") or "")
            if k in pri:
                rank = pri.index(k)
            else:
                rank = len(pri) + 100
            # later runs should win if same rank: rely on list order (append order),
            # so we prefer the last one by using <= and overwriting.
            if rank <= best_rank:
                best_rank = rank
                best = it.get("image_path")

        return best

    # ---- IMPORTANT: adapt to your JSON schema (ppt.*) ----
    def _ppt(slide: dict) -> dict:
        return (slide.get("ppt") or {}) if isinstance(slide, dict) else {}

    def _extract_cover_text(slide: dict) -> tuple[str, str, str]:
        p = _ppt(slide)
        title_line = (p.get("title") or "").strip()
        subtitle_line = (p.get("subtitle") or "").strip()
        footer_text = ""  # your JSON currently doesn't store footer; keep blank or derive if needed
        return title_line, subtitle_line, footer_text

    def _extract_left_body_text(slide: dict, max_chars: int = 320) -> str:
        p = _ppt(slide)
        text = (p.get("body_text") or "").replace("\\n", "\n").strip()
        if not text:
            return "（本文未設定）"
        if len(text) > max_chars:
            text = text[:max_chars].rstrip() + "…"
        return text

    # ------------------------------------------------------------
    # PPTX Settings (16:9)
    # ------------------------------------------------------------
    prs = Presentation()
    prs.slide_width = Inches(13.333)   # 16:9
    prs.slide_height = Inches(7.5)
    blank_layout = prs.slide_layouts[6]

    # Palette
    BCAP_DARK = "#0A211A"
    BCAP_ACCENT = "#0AC985"
    DARK_RGB = _hex_to_rgb(BCAP_DARK)
    ACCENT_RGB = _hex_to_rgb(BCAP_ACCENT)

    W = prs.slide_width
    H = prs.slide_height

    # --- Layout numbers from your requirement ---
    # Right diagram asset (15cm x 13cm), top-left at (x=17.5cm, y=3cm)
    DIAG_X = Cm(17.5)
    DIAG_Y = Cm(3.0)
    DIAG_W = Cm(15.0)
    DIAG_H = Cm(13.0)

    # Title position (top-left)
    TITLE_X = Cm(2.0)
    TITLE_Y = Cm(1.2)
    TITLE_W = W - Cm(3.0)
    TITLE_H = Cm(1.6)

    # Left body (single textbox)
    BODY_X = Cm(2.0)
    BODY_Y = Cm(3.2)
    BODY_W = Cm(14.5)          # leaves space before the diagram at x=17.5cm
    BODY_H = H - Cm(4.0)

    # Footer (cover only) - keep reserved (optional)
    FOOTER_X = Cm(2.0)
    FOOTER_Y = H - Cm(1.2)
    FOOTER_W = W - Cm(3.0)
    FOOTER_H = Cm(0.8)

    # ------------------------------------------------------------
    # Build maps / get slides
    # ------------------------------------------------------------
    spec = confirmed_slide_specifications
    slides = (spec.get("slides") or []) or []
    if not slides:
        raise RuntimeError("confirmed_slide_specifications.slides is empty")

    image_map = _build_image_map(generated_slide_images)

    # ------------------------------------------------------------
    # Build PPTX
    # ------------------------------------------------------------
    for s in slides:
        sn = _safe_int(s.get("slide_number"), 0)
        p = _ppt(s)
        stitle = (p.get("title") or "").strip()

        slide = prs.slides.add_slide(blank_layout)

        # pick image
        img_path = _pick_image_for_slide(sn, s, image_map)

        if sn == 1:
            # Cover background image (full slide)
            if img_path:
                slide.shapes.add_picture(img_path, 0, 0, width=W, height=H)

            cover_title, cover_sub, cover_footer = _extract_cover_text(s)

            # Title textbox
            tb = slide.shapes.add_textbox(TITLE_X, TITLE_Y, TITLE_W, TITLE_H)
            tf = tb.text_frame
            tf.clear()
            tf.word_wrap = True

            p0 = tf.paragraphs[0]
            p0.text = ""
            r0 = p0.add_run()
            r0.text = cover_title or stitle or "Untitled"
            _set_run_font_meiryo(r0, size_pt=36, bold=True, color=DARK_RGB)

            if cover_sub:
                p2 = tf.add_paragraph()
                p2.text = ""
                r2 = p2.add_run()
                r2.text = cover_sub
                _set_run_font_meiryo(r2, size_pt=24, bold=False, color=DARK_RGB)

            if cover_footer:
                fb = slide.shapes.add_textbox(FOOTER_X, FOOTER_Y, FOOTER_W, FOOTER_H)
                ff = fb.text_frame
                ff.clear()
                fp = ff.paragraphs[0]
                fp.text = ""
                fr = fp.add_run()
                fr.text = cover_footer
                _set_run_font_meiryo(fr, size_pt=14, bold=False, color=DARK_RGB)

        else:
            # Title
            tb = slide.shapes.add_textbox(TITLE_X, TITLE_Y, TITLE_W, TITLE_H)
            tf = tb.text_frame
            tf.clear()
            p0 = tf.paragraphs[0]
            p0.text = ""
            r0 = p0.add_run()
            r0.text = stitle or f"Slide {sn}"
            _set_run_font_meiryo(r0, size_pt=36, bold=True, color=DARK_RGB)

            # Accent bar (optional)
            try:
                accent = slide.shapes.add_shape(
                    MSO_SHAPE.RECTANGLE,
                    Cm(1.3), TITLE_Y + Cm(0.15),
                    Cm(0.25), Cm(1.0)
                )
                accent.fill.solid()
                accent.fill.fore_color.rgb = ACCENT_RGB
                accent.line.fill.background()
            except Exception:
                pass

            # Body (single textbox)
            body_text = _extract_left_body_text(s, max_chars=320)
            body_box = slide.shapes.add_textbox(BODY_X, BODY_Y, BODY_W, BODY_H)
            btf = body_box.text_frame
            btf.clear()
            btf.word_wrap = True

            # Keep one textbox but allow multiple paragraphs
            lines = [ln.strip() for ln in (body_text or "").split("\n") if ln.strip()]
            if not lines:
                lines = ["（本文未設定）"]

            b0 = btf.paragraphs[0]
            b0.text = lines[0]
            _set_paragraph_font_meiryo(b0, size_pt=24, bold=False, color=DARK_RGB)

            for ln in lines[1:]:
                bp = btf.add_paragraph()
                bp.text = ln
                _set_paragraph_font_meiryo(bp, size_pt=24, bold=False, color=DARK_RGB)

            # Right diagram image
            if img_path:
                slide.shapes.add_picture(img_path, DIAG_X, DIAG_Y, width=DIAG_W, height=DIAG_H)

    # ------------------------------------------------------------
    # Save
    # ------------------------------------------------------------
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    pptx_filename = f"presentation_editable_{ts}.pptx"
    pptx_path = OUTPUT_DIR / pptx_filename
    prs.save(str(pptx_path))

    final_pptx_path = str(pptx_path)

    print("=" * 60)
    print("✅ PPTX 組み立て完了")
    print(f"  出力ファイル: {pptx_filename}")
    print(f"  保存先: {final_pptx_path}")
    print()
    print("次のアクション: PowerPoint / Keynote で開いて編集してください。")


✅ PPTX 組み立て完了
  出力ファイル: presentation_editable_20260205_145932.pptx
  保存先: output/presentation_editable_20260205_145932.pptx

次のアクション: PowerPoint / Keynote で開いて編集してください。


In [55]:
# ============================================================
# Cell 11 — Final output verification and export (PPTX-first, hardened)
# ============================================================
# Overview:
#   Final verification for the generated PPTX, workflow summary,
#   and export utilities for metadata + artifact listing.
#
# Inputs / Outputs:
#   Inputs:  final_pptx_path (str, from Cell 10)
#            confirmed_slide_specifications (dict, from Cell 06)
#            generated_slide_images (list, from Cell 08)
#            meeting_context (dict, from Cell 03)
#   Outputs: Summary report + export button
#            workflow_metadata_YYYYMMDD_HHMMSS.json in output/
#
# Notes:
#   - Verifies PPTX existence + readability using python-pptx
#   - Counts actual PPTX slides
#   - Counts actual successful images (status=success + file exists)
#   - Exports a single JSON metadata file for archival
#

import json
from pathlib import Path
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output

# Optional PPTX check
try:
    from pptx import Presentation
    _HAVE_PPTX = True
except Exception:
    _HAVE_PPTX = False

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def verify_pptx_output(pptx_path: str) -> dict:
    """
    Verify PPTX file exists and collect stats: size, readable, slide_count.
    """
    result = {
        'exists': False,
        'size_kb': 0.0,
        'readable': False,
        'slide_count': 0,
        'error': None,
    }

    if not pptx_path:
        result['error'] = "pptx_path is empty"
        return result

    p = Path(pptx_path)
    if not p.exists():
        result['error'] = "file not found"
        return result

    result['exists'] = True
    result['size_kb'] = p.stat().st_size / 1024.0

    if _HAVE_PPTX:
        try:
            prs = Presentation(str(p))
            result['slide_count'] = len(prs.slides)
            result['readable'] = True
        except Exception as e:
            result['readable'] = False
            result['error'] = f"{type(e).__name__}: {e}"
    else:
        result['readable'] = False
        result['error'] = "python-pptx not installed; cannot verify readability/slide count"

    return result

def _successful_images() -> list[dict]:
    """
    Return list of images that are status=success and file exists.
    """
    imgs = []
    if 'generated_slide_images' not in globals():
        return imgs

    for img in (generated_slide_images or []):
        if img.get("status") != "success":
            continue
        path = img.get("image_path")
        if not path:
            continue
        if Path(path).exists():
            imgs.append(img)

    def _k(d):
        try:
            return int(d.get("slide_number", 0))
        except Exception:
            return 0
    return sorted(imgs, key=_k)

def collect_workflow_statistics() -> dict:
    stats = {}

    # Meeting context
    mc = meeting_context if 'meeting_context' in globals() else {}
    stats['meeting_configured'] = bool(mc)
    stats['counterpart'] = mc.get('counterpart_name', 'N/A')
    stats['user_name'] = mc.get('user_name', 'N/A')
    stats['meeting_goal'] = mc.get('meeting_goal', 'N/A')

    # Outline
    if 'presentation_outline' in globals():
        stats['outline_confirmed'] = True
        stats['outline_length'] = len(presentation_outline or "")
    else:
        stats['outline_confirmed'] = False
        stats['outline_length'] = 0

    # Slide specs
    if 'confirmed_slide_specifications' in globals():
        stats['json_confirmed'] = True
        slides = confirmed_slide_specifications.get('slides', []) or []
        stats['total_slides_in_json'] = len(slides)
        md = confirmed_slide_specifications.get('metadata', {}) or {}
        stats['deck_title'] = md.get('title', 'N/A')
        stats['deck_audience'] = md.get('audience', 'N/A')
        stats['deck_author'] = md.get('author', 'N/A')
    else:
        stats['json_confirmed'] = False
        stats['total_slides_in_json'] = 0
        stats['deck_title'] = 'N/A'
        stats['deck_audience'] = 'N/A'
        stats['deck_author'] = 'N/A'

    # Images
    ok_imgs = _successful_images()
    stats['images_success'] = len(ok_imgs)
    if 'generated_slide_images' in globals():
        stats['images_error'] = len([i for i in (generated_slide_images or []) if i.get("status") == "error"])
    else:
        stats['images_error'] = 0

    # PPTX
    if 'final_pptx_path' in globals():
        pptx_info = verify_pptx_output(final_pptx_path)
        stats['pptx_created'] = pptx_info['exists']
        stats['pptx_readable'] = pptx_info['readable']
        stats['pptx_size_kb'] = pptx_info['size_kb']
        stats['pptx_slides'] = pptx_info['slide_count']
        stats['pptx_error'] = pptx_info['error']
    else:
        stats['pptx_created'] = False
        stats['pptx_readable'] = False
        stats['pptx_size_kb'] = 0.0
        stats['pptx_slides'] = 0
        stats['pptx_error'] = "final_pptx_path not found"

    return stats

def list_artifacts() -> dict:
    """
    List key artifacts saved in OUTPUT_DIR.
    """
    try:
        out_dir = OUTPUT_DIR
    except Exception:
        out_dir = None

    if not out_dir or not Path(out_dir).exists():
        return {"output_dir": str(out_dir), "files": []}

    p = Path(out_dir)
    files = sorted([x for x in p.glob("*") if x.is_file()], key=lambda x: x.stat().st_mtime, reverse=True)
    return {"output_dir": str(p), "files": [f.name for f in files]}

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------
stats = collect_workflow_statistics()

print('=' * 70)
print('📊 プレゼンテーション生成ワークフロー — 最終検証レポート（PPTX）')
print('=' * 70)
print()

print('【ワークフロー完了状況】')
print(f"  ✓ ミーティング情報入力: {'完了' if stats['meeting_configured'] else '未完了'}")
print(f"  ✓ アウトライン生成: {'完了' if stats['outline_confirmed'] else '未完了'}")
print(f"  ✓ スライドJSON確定: {'完了' if stats['json_confirmed'] else '未完了'}")
print(f"  ✓ 画像生成: success={stats['images_success']} / error={stats['images_error']}")
print(f"  ✓ PPTX組み立て: {'完了' if stats['pptx_created'] else '未完了'}")
print()

if 'final_pptx_path' in globals():
    pptx_info = verify_pptx_output(final_pptx_path)
    print('【最終PPTX検証】')
    print(f"  ファイル名: {Path(final_pptx_path).name}")
    print(f"  保存先: {final_pptx_path}")
    print(f"  存在確認: {'✅ OK' if pptx_info['exists'] else '❌ NG'}")
    print(f"  読み取り: {'✅ OK' if pptx_info['readable'] else '❌ NG'}")
    if pptx_info['slide_count']:
        print(f"  スライド数(PPTX実測): {pptx_info['slide_count']}")
    else:
        print(f"  スライド数(PPTX実測): N/A")
    print(f"  ファイルサイズ: {pptx_info['size_kb']:.1f} KB")
    if pptx_info.get('error'):
        print(f"  備考: {pptx_info['error']}")
    print()
else:
    print('⚠️  最終PPTXが見つかりません (Cell 10を実行してください)\n')

# ------------------------------------------------------------
# Export Metadata
# ------------------------------------------------------------
def export_complete_metadata() -> str:
    ok_imgs = _successful_images()
    artifacts = list_artifacts()

    payload = {
        "workflow_timestamp": datetime.now().isoformat(),
        "statistics": stats,
        "meeting_context": meeting_context if 'meeting_context' in globals() else {},
        "presentation_outline": presentation_outline if 'presentation_outline' in globals() else "",
        "confirmed_slide_specifications": confirmed_slide_specifications if 'confirmed_slide_specifications' in globals() else {},
        "final_pptx_path": final_pptx_path if 'final_pptx_path' in globals() else None,
        "generated_images": [
            {
                "slide_number": i.get("slide_number"),
                "title": i.get("title"),
                "filename": i.get("filename"),
                "image_path": i.get("image_path"),
                "run_timestamp": i.get("run_timestamp"),
                "kind": i.get("kind"),
                "status": i.get("status"),
                "background": i.get("background"),
            }
            for i in ok_imgs
        ],
        "artifacts": artifacts,
    }

    export_path = OUTPUT_DIR / f"workflow_metadata_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(export_path, 'w', encoding='utf-8') as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    return str(export_path)

w_export_btn = widgets.Button(
    description='📦 メタデータをエクスポート',
    button_style='info',
    layout=widgets.Layout(width='240px')
)
w_export_output = widgets.Output()

def on_export_clicked(_):
    with w_export_output:
        clear_output()
        try:
            path = export_complete_metadata()
            print(f'✅ エクスポート完了: {Path(path).name}')
            print(f'   保存先: {path}')
        except Exception as e:
            print(f'❌ エクスポートエラー: {type(e).__name__}: {e}')

w_export_btn.on_click(on_export_clicked)

display(widgets.VBox([
    widgets.HTML('<h3>📤 メタデータエクスポート</h3>'),
    w_export_btn,
    w_export_output
]))

# ------------------------------------------------------------
# Artifact listing
# ------------------------------------------------------------
print('=' * 70)
print('📁 output/ にある成果物（最新順）')
print('=' * 70)

art = list_artifacts()
print(f"出力ディレクトリ: {art.get('output_dir')}")
files = art.get("files", [])
for f in files[:25]:
    print(f"  - {f}")
if len(files) > 25:
    print(f"  ...他 {len(files)-25} 件")

print()
print('=' * 70)
print('✅ 検証完了')
print('=' * 70)
print()
print('次のアクション:')
print('  1) 生成されたPPTXを PowerPoint / Keynote で開いて確認')
print('  2) 必要なら「メタデータをエクスポート」でJSON保存')
print('  3) 問題があれば Cell 06でJSON編集 → Cell 07/08/10を再実行')


📊 プレゼンテーション生成ワークフロー — 最終検証レポート（PPTX）

【ワークフロー完了状況】
  ✓ ミーティング情報入力: 完了
  ✓ アウトライン生成: 完了
  ✓ スライドJSON確定: 完了
  ✓ 画像生成: success=10 / error=0
  ✓ PPTX組み立て: 完了

【最終PPTX検証】
  ファイル名: presentation_editable_20260205_144725.pptx
  保存先: output/presentation_editable_20260205_144725.pptx
  存在確認: ✅ OK
  読み取り: ✅ OK
  スライド数(PPTX実測): 10
  ファイルサイズ: 4911.7 KB



📁 output/ にある成果物（最新順）
出力ディレクトリ: output
  - presentation_editable_20260205_144725.pptx
  - presentation_editable_20260205_134424.pptx
  - generated_images_meta_20260205_133921_whitebg.json
  - slide_20260205_133921_010_diagram_asset.png
  - gemini_raw_20260205_133921_010_diagram_asset.png
  - slide_20260205_133921_009_diagram_asset.png
  - gemini_raw_20260205_133921_009_diagram_asset.png
  - slide_20260205_133921_008_diagram_asset.png
  - gemini_raw_20260205_133921_008_diagram_asset.png
  - slide_20260205_133921_007_diagram_asset.png
  - gemini_raw_20260205_133921_007_diagram_asset.png
  - slide_20260205_133921_006_diagram_asset.png
  - gemini_raw_20260205_133921_006_diagram_asset.png
  - slide_20260205_133921_005_diagram_asset.png
  - gemini_raw_20260205_133921_005_diagram_asset.png
  - slide_20260205_133921_004_diagram_asset.png
  - gemini_raw_20260205_133921_004_diagram_asset.png
  - slide_20260205_133921_003_diagram_asset.png
  - gemini_raw_20260205_133921_003_diagram_asset.png
  - 